# Cross-dataset raw persona audit

## Excluded persona attributes

Attributes intentionally omitted from `persona_spec.yaml`:

| Attribute | Reason for exclusion |
| --- | --- |
| `census_division` | Excluded as a separate attribute (only natively present in `issp`). ISSP's 9 US Census divisions are kept directly as union levels inside `region` without collapsing. |
| `interview_language` | Excluded (`issp` `INTLANG`, `wvs7` `S_INTLANGUAGE`): survey-administration technical variable (~98% English), not a respondent persona trait. |
| `birth_year` | Excluded (`issp` `BIRTH`, `wvs7` `Q261`): 100% redundant with locked `age`, and pairing `age` with exact `birth_year` leaks survey wave year. |
| `US_DEGR` | Excluded (`issp` 2015-2019): 100% redundant with locked `education` (`EDULEVEL`/`DEGREE`). |
| `X003R` | Excluded (`wvs7`): 6-bin age recode, 100% redundant with locked exact `age` (`Q262`). |
| `EDUCYRS` | Excluded (`issp` years of full-time schooling): covered by locked `education`. |
| `TOPBOT` | Excluded (`issp` 1-10 subjective social standing scale): omitted per user review. |
| `WORK` | Excluded (`issp` paid work history): largely redundant with `employment_status`; omitted per user review. |
| `WRKHRS` | Excluded (`issp` weekly hours worked): omitted per user review. |
| `EMPREL` | Excluded (`issp` employment relationship): omitted per user review. |
| `WRKSUP` | Excluded (`issp` supervises other employees): omitted per user review. |
| `NSUP` | Excluded (`issp` number of employees supervised): omitted per user review. |
| `TYPORG1` | Excluded (`issp` for-profit vs non-profit organization): omitted per user review. |
| `PARTLIV` | Excluded (`issp` living in steady partnership): omitted per user review. |
| `SPWORK` | Excluded (`issp` spouse/partner paid work history): omitted per user review. |
| `SPWRKHRS` | Excluded (`issp` spouse/partner weekly hours worked): omitted per user review. |
| `SPEMPREL` | Excluded (`issp` spouse/partner employment relationship): omitted per user review. |
| `SPWRKSUP` | Excluded (`issp` spouse/partner supervises other employees): omitted per user review. |
| `SPISCO08` | Excluded (`issp` spouse/partner ISCO-08 occupation): omitted per user review. |
| `HHCHILDR` | Excluded (`issp` school-age children in household): omitted per user review. |
| `HHTODD` | Excluded (`issp` toddlers below school entry age in household): omitted per user review. |
| `HHADULT` | Excluded (`issp` adults in household): omitted per user review. |
| `H_SETTLEMENT` | Excluded (`wvs7` settlement type): non-US administrative wording duplicating `town_size` and `urban_rural`; omitted per user review. |
| `VENDING_BEV` | Excluded (`grummon2023` vending machine beverage frequency): narrow study-specific shopping frequency (~70% floor, 86% identical to `VENDING_SNACK`); omitted per user review. |
| `VENDING_SNACK` | Excluded (`grummon2023` vending machine snack frequency): narrow study-specific shopping frequency (~71% floor, 86% identical to `VENDING_BEV`); omitted per user review. |
| `covid__party_lean`, `kreps2021__party_lean` | Excluded (`covid` & `kreps2021` party lean): fielded to all respondents without skip logic, contradicting `party3` for declared partisans (70 cross-party Democrat/Republican rows in `covid`, 43 in `kreps2021`); omitted per user review. |
| `berke2023__Q_v_1` | Excluded (`berke2023` hypothetical prescription delivery willingness): conjoint block screener that gates `CHOICE_v1_*` (0/392 `No` saw block); omitted per user review. |
| `berke2023__Q_v_2` | Excluded (`berke2023` hypothetical liquor store delivery willingness): conjoint block screener that gates `CHOICE_v2_*` (0/1,334 `No` saw block); omitted per user review. |
| `kriner2026__urban_rural_least` | Excluded (`kriner2026` least-descriptive urban/rural companion item): negative-ranking companion to already-locked `urban_rural` (`urban_rural_best`: Suburban 476, Urban 320, Rural 204 vs. least: Rural 498, Urban 351, Suburban 151); omitted per user review. |
| `kriner2026__climate_change_old` | Excluded (`kriner2026` climate-change policy opinion): domain-specific post-conjoint attitude (`catalog_idx = 119`) kept as a target survey question in `kriner2026_US_long.parquet` (consistent with `wu2025` `warmingbelief` and `covid` `vaccine_safety`); omitted per user review. |
| `kriner2026__saf_familiar_old` | Excluded (`kriner2026` SAF familiarity): study-specific item embedded inside the SAF conjoint introduction block (`catalog_idx = 43`, 54.4% at floor), kept as a target survey question in `kriner2026_US_long.parquet`; omitted per user review. |
| `kriner2026__sci_know_self` | Excluded (`kriner2026` self-rated science knowledge): post-quiz/post-conjoint subjective 7-point self-rating with bare numeric middle labels, kept as a target survey question in `kriner2026_US_long.parquet`; omitted per user review. |
| `kantorowicz2026__risk_aversion_dummy` | Excluded (`kantorowicz2026` behavioral risk-task classification): undocumented, unused in paper or replication script, no prompt or cutoff; omitted per user review. |
| `kantorowicz2026__analytical_thinking_dummy` | Excluded (`kantorowicz2026` binarized 3-item CRT score): undocumented threshold and no raw CRT item answers in replication archive; omitted per user review. |
| `kantorowicz2026__consistent_switch` | Excluded (`kantorowicz2026` risk-task switching consistency flag): undocumented binary flag with no prompt or definition; omitted per user review. |
| `magistro2025__education_bin` | Excluded (`magistro2025` derived binary recode): deterministic 2-level collapse of already-locked `education` (`College` vs `No college`). |
| `magistro2025__gender_bin` | Excluded (`magistro2025` derived binary recode): lossy copy of already-locked `gender` that maps `Other` (79) to `<NA>`. |
| `forsythe2023__Clean Vehicle Type` | Excluded (`forsythe2023` conjoint block screener): routes respondents to the passenger-car vs SUV choice experiment (`demo-car` vs `demo-suv`), not a general persona attribute. |


In [1]:
%load_ext autoreload
%autoreload 2

import importlib
import io
import json
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display
from pandas.io.stata import StataReader

CODE_DIR = Path("../..").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from data import persona_loaders, persona_spec
importlib.reload(persona_spec)
importlib.reload(persona_loaders)

_LIVE_STATE: dict[str, object] = {"sig": None, "specs": None, "raw_cache": {}}


def _ensure_fresh_persona_code() -> dict:
    """Reload persona_spec + persona_loaders automatically whenever code or YAML changes on disk."""
    spec_yaml = Path(persona_spec.__file__).resolve().with_name("persona_spec.yaml")
    sig = (
        Path(persona_spec.__file__).stat().st_mtime_ns,
        Path(persona_loaders.__file__).stat().st_mtime_ns,
        spec_yaml.stat().st_mtime_ns,
    )
    if _LIVE_STATE["sig"] != sig:
        importlib.reload(persona_spec)
        importlib.reload(persona_loaders)
        _LIVE_STATE["sig"] = sig
        _LIVE_STATE["specs"] = persona_spec.load_persona_spec()
        _LIVE_STATE["raw_cache"] = {}
    return _LIVE_STATE["specs"]


def _format_series_counts(series: pd.Series, col_label: str, max_rows: int | None = None) -> pd.DataFrame:
    s = series.where(series.notna(), "<NA>").astype(str)
    counts = s.value_counts()
    na_count = int(counts["<NA>"]) if "<NA>" in counts.index else 0
    body = (counts.drop(index="<NA>") if "<NA>" in counts.index else counts).sort_values(ascending=False)
    if max_rows is not None and len(body) > max_rows:
        other_n = int(body.iloc[max_rows:].sum())
        other_k = len(body) - max_rows
        body = pd.concat([body.iloc[:max_rows], pd.Series({f"... ({other_k} more values)": other_n})])
    out = pd.concat([body, pd.Series({"<NA>": na_count, "All": int(len(series))})])
    df_out = out.to_frame(name="count")
    df_out["%"] = (df_out["count"] / len(series) * 100).round(1)
    df_out.index.name = col_label
    return df_out


def show_raw_counts(raw_df: pd.DataFrame, col: str, max_rows: int | None = None) -> pd.DataFrame:
    """Show value counts directly from an untouched source survey DataFrame."""
    if col not in raw_df.columns:
        raise KeyError(f"Raw column {col!r} not found in source DataFrame")
    return _format_series_counts(raw_df[col], f"raw: {col}", max_rows=max_rows)


def show_persona_counts(
    dataset: str, col: str, standardized: bool = True, max_rows: int | None = None
) -> pd.DataFrame:
    """Show live post-loader (and optionally post-spec) counts, auto-reloading when code/YAML changes."""
    specs = _ensure_fresh_persona_code()
    raw_cache: dict[str, pd.DataFrame] = _LIVE_STATE["raw_cache"]
    if dataset not in raw_cache:
        raw_cache[dataset] = persona_loaders.RAW_LOADERS[dataset]()
    raw_df = raw_cache[dataset]
    if col not in raw_df.columns:
        raise KeyError(f"Column {col!r} not produced by {dataset} persona loader")
    if standardized:
        if col not in specs:
            raise KeyError(
                f"Column {col!r} is not in persona_spec.yaml (pass standardized=False for unmapped/excluded cols)"
            )
        std_df = persona_spec.standardize_dataframe(
            raw_df[["CASEID", col]].copy(), dataset=dataset, specs={col: specs[col]}
        )
        return _format_series_counts(std_df[col], f"{dataset} (persona/std): {col}", max_rows=max_rows)
    return _format_series_counts(raw_df[col], f"{dataset} (persona/loader): {col}", max_rows=max_rows)


def show_value_counts(df, col):
    present_ds = [
        ds for ds in dataset_order
        if df.loc[df["dataset"] == ds, col].notna().any()
    ]
    sub = df.loc[df["dataset"].isin(present_ds), ["dataset", col]].copy()
    sub[col] = sub[col].where(sub[col].notna(), "<NA>").astype(str)

    table = pd.crosstab(sub[col], sub["dataset"], margins=True)
    table = table[present_ds + ["All"]]

    has_na = "<NA>" in table.index
    drop_idx = ["All", "<NA>"] if has_na else ["All"]
    body = table.drop(index=drop_idx).sort_values("All", ascending=False)
    na_row = table.loc[["<NA>"]] if has_na else pd.DataFrame(0, index=["<NA>"], columns=table.columns)
    return pd.concat([body, na_row, table.loc[["All"]]])


def show_shared_counts(col: str, max_rows: int | None = None) -> pd.DataFrame:
    """Show live cross-dataset standardized value counts for a shared attribute (auto-reloads code/YAML)."""
    specs = _ensure_fresh_persona_code()
    if col not in specs:
        raise KeyError(f"Column {col!r} is not an active attribute in persona_spec.yaml")
    raw_cache: dict[str, pd.DataFrame] = _LIVE_STATE["raw_cache"]
    frames = []
    for ds in dataset_order:
        if ds not in raw_cache:
            raw_cache[ds] = persona_loaders.RAW_LOADERS[ds]()
        raw_df = raw_cache[ds]
        if col not in raw_df.columns:
            continue
        std_df = persona_spec.standardize_dataframe(
            raw_df[["CASEID", col]].copy(), dataset=ds, specs={col: specs[col]}
        )
        if std_df[col].notna().any():
            frames.append(std_df[[col]].assign(dataset=ds))
    if not frames:
        raise ValueError(f"No dataset has non-null standardized values for {col!r}")
    sub = pd.concat(frames, ignore_index=True)
    present_ds = [ds for ds in dataset_order if ds in set(sub["dataset"])]
    sub[col] = sub[col].where(sub[col].notna(), "<NA>").astype(str)
    table = pd.crosstab(sub[col], sub["dataset"], margins=True)[present_ds + ["All"]]
    has_na = "<NA>" in table.index
    drop_idx = ["All", "<NA>"] if has_na else ["All"]
    body = table.drop(index=drop_idx).sort_values("All", ascending=False)
    if max_rows is not None and len(body) > max_rows:
        other = body.iloc[max_rows:].sum(axis=0).to_frame().T
        other.index = [f"... ({len(body) - max_rows} more values)"]
        body = pd.concat([body.iloc[:max_rows], other])
    na_row = table.loc[["<NA>"]] if has_na else pd.DataFrame(0, index=["<NA>"], columns=table.columns)
    out = pd.concat([body, na_row, table.loc[["All"]]])
    out.index.name = f"std: {col}"
    return out


all_personas = persona_loaders.load_raw_all_personas()
std_personas = persona_loaders.load_all_personas()
dataset_order = list(persona_loaders.RAW_LOADERS.keys())
persona_cols = [c for c in all_personas.columns if c not in {"dataset", "CASEID"}]
ds_counts = {c: int(all_personas.loc[all_personas[c].notna(), "dataset"].nunique()) for c in persona_cols}
shared_cols = [c for c, n in ds_counts.items() if n >= 2]
unique_cols = [c for c, n in ds_counts.items() if n == 1]
unmapped_cols = [c for c in persona_cols if "__" in c]

print(f"Stacked all_personas (raw): {len(all_personas):,} rows x {all_personas.shape[1]} cols")
print(f"Stacked std_personas (canonical): {len(std_personas):,} rows x {std_personas.shape[1]} cols")
print(f"Shared ({len(shared_cols)}): {shared_cols}")
print(f"Unique ({len(unique_cols)}): {unique_cols}")
print(f"Unmapped candidates ({len(unmapped_cols)}): {unmapped_cols}")
all_personas["dataset"].value_counts().reindex(dataset_order)

Stacked all_personas (raw): 72,402 rows x 128 cols
Stacked std_personas (canonical): 72,402 rows x 82 cols
Shared (35): ['country', 'gender', 'age', 'region', 'family_income', 'religion', 'religious_attendance', 'political_party', 'marital_status', 'employment_status', 'household_size', 'urban_rural', 'voted_last_election', 'union_member', 'employer_sector', 'birth_year', 'spouse_employment', 'occupation', 'education', 'interview_language', 'spouse_education', 'political_ideology', 'race', 'citizen', 'state', 'income', 'children_in_household', 'hispanic', 'sexual_orientation', 'residence_type', 'health_insurance', 'evangelical_christian', 'flu_vaccination_frequency', 'trump_approval', 'nutrition_label_use']
Unique (91): ['EDUCYRS', 'US_DEGR', 'personal_income', 'WORK', 'WRKHRS', 'EMPREL', 'WRKSUP', 'NSUP', 'TYPORG1', 'PARTLIV', 'SPWORK', 'SPWRKHRS', 'SPEMPREL', 'SPWRKSUP', 'SPISCO08', 'HHCHILDR', 'HHTODD', 'TOPBOT', 'ancestry', 'HHADULT', 'town_size', 'H_SETTLEMENT', 'X003R', 'immigran

dataset
issp               14365
twin2k              2058
wvs7                2596
subpop                22
hints7              7278
shed2025           12934
covid               1971
kreps2021           1096
grummon2023         7945
grummon2025         5636
berke2023           3715
kriner2026          1000
wu2025              1862
kantorowicz2026     2242
magistro2025        6056
forsythe2023        1626
Name: count, dtype: int64

In [2]:
all_personas.columns[:20]

Index(['dataset', 'CASEID', 'country', 'gender', 'age', 'region',
       'family_income', 'religion', 'religious_attendance', 'political_party',
       'marital_status', 'employment_status', 'household_size', 'urban_rural',
       'voted_last_election', 'union_member', 'employer_sector', 'birth_year',
       'spouse_employment', 'occupation'],
      dtype='str')

## 1. Cross-dataset value counts — all 33 active shared persona attributes

In [ ]:
show_shared_counts("country")

In [ ]:
show_shared_counts("gender")

In [ ]:
show_shared_counts("age")

In [ ]:
show_shared_counts("region")

In [ ]:
show_shared_counts("state", max_rows=25)

In [ ]:
show_shared_counts("education")

In [ ]:
show_shared_counts("family_income", max_rows=30)

In [ ]:
show_shared_counts("income")

In [ ]:
show_shared_counts("race", max_rows=25)

In [ ]:
show_shared_counts("hispanic")

In [ ]:
show_shared_counts("religion")

In [ ]:
show_shared_counts("religious_attendance")

In [ ]:
show_shared_counts("evangelical_christian")

In [ ]:
show_shared_counts("political_party")

In [ ]:
show_shared_counts("political_ideology")

In [ ]:
show_shared_counts("voted_last_election")

In [ ]:
show_shared_counts("trump_approval")

In [ ]:
show_shared_counts("marital_status")

In [ ]:
show_shared_counts("employment_status")

In [ ]:
show_shared_counts("employer_sector")

In [ ]:
show_shared_counts("occupation", max_rows=20)

In [ ]:
show_shared_counts("union_member")

In [ ]:
show_shared_counts("household_size")

In [ ]:
show_shared_counts("children_in_household")

In [ ]:
show_shared_counts("urban_rural")

In [ ]:
show_shared_counts("residence_type")

In [ ]:
show_shared_counts("citizen")

In [ ]:
show_shared_counts("sexual_orientation")

In [ ]:
show_shared_counts("health_insurance")

In [ ]:
show_shared_counts("flu_vaccination_frequency")

In [ ]:
show_shared_counts("nutrition_label_use")

In [ ]:
show_shared_counts("spouse_education")

In [ ]:
show_shared_counts("spouse_employment")

## 2. Per-dataset persona audit (shared & dataset-specific)

### 2.1 ISSP (complete)

In [ ]:
_issp_frames = []
for _yr in range(2015, 2025):
    _dta = list((persona_loaders.DATA_DIR / f"issp{_yr}" / "raw").glob("*.dta"))[0]
    _vl, _c2l = persona_loaders._read_stata_value_labels(_dta)
    _want = [*persona_loaders.ISSP_RAW_CANDIDATES, *([persona_loaders.ISSP_IDEOLOGY_VAR[_yr]] if _yr in persona_loaders.ISSP_IDEOLOGY_VAR else [])]
    _present = [c for c in dict.fromkeys(_want) if c in _c2l]
    _df = pd.read_stata(_dta, columns=sorted({"CASEID", "c_alphan", *_present}), convert_categoricals=False)
    _us = _df[_df["c_alphan"] == "US"].reset_index(drop=True)
    _wave = pd.DataFrame({c: persona_loaders._decode_stata_col(_us[c], c, _vl, _c2l) for c in _present})
    _issp_frames.append(_wave)
issp_raw = pd.concat(_issp_frames, ignore_index=True)
issp = all_personas[all_personas["dataset"] == "issp"].dropna(axis=1, how="all")
issp.head(2)


#### Shared attributes

In [ ]:
# Country (c_alphan) -> country
display(show_raw_counts(issp_raw, "c_alphan"))
display(show_persona_counts("issp", "country"))


In [ ]:
# Gender (SEX) -> gender
display(show_raw_counts(issp_raw, "SEX"))
display(show_persona_counts("issp", "gender"))


In [ ]:
# Age in years (AGE) -> age
display(show_raw_counts(issp_raw, "AGE", max_rows=20))
display(show_persona_counts("issp", "age", max_rows=20))


In [ ]:
# Year of birth (BIRTH) — excluded from persona_spec.yaml (redundant with age & leaks wave year)
display(show_raw_counts(issp_raw, "BIRTH", max_rows=20))
display(show_persona_counts("issp", "birth_year", standardized=False, max_rows=20))


In [ ]:
# US Census division (US_REG) -> region
display(show_raw_counts(issp_raw, "US_REG"))
display(show_persona_counts("issp", "region"))


In [ ]:
# Urban/rural self-description (URBRURAL) -> urban_rural
display(show_raw_counts(issp_raw, "URBRURAL"))
display(show_persona_counts("issp", "urban_rural"))


In [ ]:
# Education (EDULEVEL in 2020-2024; DEGREE in 2015-2019) -> education
display(show_raw_counts(issp_raw, "EDULEVEL"))
display(show_raw_counts(issp_raw, "DEGREE"))
display(show_persona_counts("issp", "education"))


In [ ]:
# Spouse/partner education (SPEDULEV) -> spouse_education
display(show_raw_counts(issp_raw, "SPEDULEV"))
display(show_persona_counts("issp", "spouse_education"))


In [ ]:
# Household/family income (US_INC) -> family_income
display(show_raw_counts(issp_raw, "US_INC", max_rows=25))
display(show_persona_counts("issp", "family_income", max_rows=25))


In [ ]:
# Religious group (RELIGGRP) -> religion
display(show_raw_counts(issp_raw, "RELIGGRP"))
display(show_persona_counts("issp", "religion"))


In [ ]:
# Religious service attendance (ATTEND) -> religious_attendance
display(show_raw_counts(issp_raw, "ATTEND"))
display(show_persona_counts("issp", "religious_attendance"))


In [ ]:
# US party identification (US_PRTY) -> political_party
display(show_raw_counts(issp_raw, "US_PRTY"))
display(show_persona_counts("issp", "political_party"))


In [ ]:
# ISSP 2023 v46 & 2024 v59 Left-Right ideology (PARTY_LR dropped) — locked -> political_ideology
display(show_raw_counts(issp_raw, "v46"))
display(show_raw_counts(issp_raw, "v59"))
display(show_persona_counts("issp", "political_ideology"))


In [ ]:
# Voted in last election (VOTE_LE) -> voted_last_election
display(show_raw_counts(issp_raw, "VOTE_LE"))
display(show_persona_counts("issp", "voted_last_election"))


In [ ]:
# Marital status (MARITAL) -> marital_status
display(show_raw_counts(issp_raw, "MARITAL"))
display(show_persona_counts("issp", "marital_status"))


In [ ]:
# Household size (HOMPOP) -> household_size
display(show_raw_counts(issp_raw, "HOMPOP"))
display(show_persona_counts("issp", "household_size"))


In [ ]:
# Main employment status (MAINSTAT) -> employment_status
display(show_raw_counts(issp_raw, "MAINSTAT"))
display(show_persona_counts("issp", "employment_status"))


In [ ]:
# Spouse/partner main employment status (SPMAINST) -> spouse_employment
display(show_raw_counts(issp_raw, "SPMAINST"))
display(show_persona_counts("issp", "spouse_employment"))


In [ ]:
# Occupation (ISCO-08) — locked -> occupation
display(show_raw_counts(issp_raw, "ISCO08", max_rows=20))
display(show_persona_counts("issp", "occupation", max_rows=20))


In [ ]:
# Public vs private employer sector (TYPORG2) -> employer_sector
display(show_raw_counts(issp_raw, "TYPORG2"))
display(show_persona_counts("issp", "employer_sector"))


In [ ]:
# Trade union membership (UNION) -> union_member
display(show_raw_counts(issp_raw, "UNION"))
display(show_persona_counts("issp", "union_member"))


In [ ]:
# Interview language (INTLANG) — excluded from persona_spec.yaml (survey admin metadata)
display(show_raw_counts(issp_raw, "INTLANG"))
display(show_persona_counts("issp", "interview_language", standardized=False))


#### Dataset-specific attributes

In [ ]:
# Years of full-time schooling — excluded (covered by locked `education`)
display(show_raw_counts(issp_raw, "EDUCYRS", max_rows=20))
display(show_persona_counts("issp", "education"))


In [ ]:
# Merged US_ETHN1 + US_ETHN2 ancestry — locked -> ancestry
display(show_raw_counts(issp_raw, "US_ETHN1", max_rows=15))
display(show_raw_counts(issp_raw, "US_ETHN2", max_rows=15))
display(show_persona_counts("issp", "ancestry", max_rows=20))


In [ ]:
# Personal income (US_RINC) — locked -> personal_income
display(show_raw_counts(issp_raw, "US_RINC", max_rows=25))
display(show_persona_counts("issp", "personal_income", max_rows=25))


In [ ]:
# Top-bottom social standing (1-10) — excluded
display(show_raw_counts(issp_raw, "TOPBOT"))


In [ ]:
# Currently, formerly, or never in paid work — excluded (covered by `employment_status`)
display(show_raw_counts(issp_raw, "WORK"))
display(show_persona_counts("issp", "employment_status"))


In [ ]:
# Hours worked weekly — excluded
display(show_raw_counts(issp_raw, "WRKHRS", max_rows=20))


In [ ]:
# Employment relationship — excluded
display(show_raw_counts(issp_raw, "EMPREL"))


In [ ]:
# Supervises other employees — excluded
display(show_raw_counts(issp_raw, "WRKSUP"))


In [ ]:
# Number of employees supervised — excluded
display(show_raw_counts(issp_raw, "NSUP", max_rows=20))


In [ ]:
# For-profit vs non-profit organization — excluded (covered by `employer_sector` TYPORG2)
display(show_raw_counts(issp_raw, "TYPORG1"))
display(show_persona_counts("issp", "employer_sector"))


In [ ]:
# Living in steady partnership — excluded (covered by `marital_status`)
display(show_raw_counts(issp_raw, "PARTLIV"))
display(show_persona_counts("issp", "marital_status"))


In [ ]:
# Spouse/partner: paid work history — excluded (covered by `spouse_employment`)
display(show_raw_counts(issp_raw, "SPWORK"))
display(show_persona_counts("issp", "spouse_employment"))


In [ ]:
# Spouse/partner: hours worked weekly — excluded
display(show_raw_counts(issp_raw, "SPWRKHRS", max_rows=20))


In [ ]:
# Spouse/partner: employment relationship — excluded
display(show_raw_counts(issp_raw, "SPEMPREL"))


In [ ]:
# Spouse/partner: supervises other employees — excluded
display(show_raw_counts(issp_raw, "SPWRKSUP"))


In [ ]:
# Spouse/partner: occupation (ISCO-08) — excluded
display(show_raw_counts(issp_raw, "SPISCO08", max_rows=20))


In [ ]:
# School-age children in household — excluded
display(show_raw_counts(issp_raw, "HHCHILDR"))


In [ ]:
# Toddlers below school entry age in household — excluded
display(show_raw_counts(issp_raw, "HHTODD"))


In [ ]:
# Adults in household — excluded (covered by `household_size`)
display(show_raw_counts(issp_raw, "HHADULT"))
display(show_persona_counts("issp", "household_size"))


### 2.2 WVS7 (complete)

In [ ]:
_wvs_dta = persona_loaders.DATA_DIR / "WVS-Wave7" / "WVS_Wave_7_United_States_Stata_v5.1.dta"
_wvs_vl, _wvs_c2l = persona_loaders._read_stata_value_labels(_wvs_dta)
_wvs_cols = list(persona_loaders.WVS_PERSONA_RAW_COLS)
_wvs_df = pd.read_stata(_wvs_dta, columns=_wvs_cols, convert_categoricals=False)
wvs7_raw = pd.DataFrame({
    c: persona_loaders._decode_stata_col(_wvs_df[c], c, _wvs_vl, _wvs_c2l)
    for c in _wvs_cols
})
wvs7 = all_personas[all_personas["dataset"] == "wvs7"].dropna(axis=1, how="all")
wvs7.head(2)


#### Shared attributes

In [ ]:
# Country (B_COUNTRY) -> country
display(show_raw_counts(wvs7_raw, "B_COUNTRY"))
display(show_persona_counts("wvs7", "country"))


In [ ]:
# US state ISO code (N_REGION_ISO) -> state
display(show_raw_counts(wvs7_raw, "N_REGION_ISO", max_rows=20))
display(show_persona_counts("wvs7", "state", max_rows=20))


In [ ]:
# Urban/rural (H_URBRURAL) -> urban_rural
display(show_raw_counts(wvs7_raw, "H_URBRURAL"))
display(show_persona_counts("wvs7", "urban_rural"))


In [ ]:
# Sex (Q260) -> gender
display(show_raw_counts(wvs7_raw, "Q260"))
display(show_persona_counts("wvs7", "gender"))


In [ ]:
# Age in years (Q262) -> age
display(show_raw_counts(wvs7_raw, "Q262", max_rows=20))
display(show_persona_counts("wvs7", "age", max_rows=20))


In [ ]:
# Year of birth (Q261) — excluded from persona_spec.yaml (redundant with age & leaks wave year)
display(show_raw_counts(wvs7_raw, "Q261", max_rows=20))
display(show_persona_counts("wvs7", "birth_year", standardized=False, max_rows=20))


In [ ]:
# Ethnic/racial group (Q290) -> race
display(show_raw_counts(wvs7_raw, "Q290"))
display(show_persona_counts("wvs7", "race"))


In [ ]:
# US citizenship (Q269) -> citizen
display(show_raw_counts(wvs7_raw, "Q269"))
display(show_persona_counts("wvs7", "citizen"))


In [ ]:
# Highest educational level (Q275) -> education
display(show_raw_counts(wvs7_raw, "Q275"))
display(show_persona_counts("wvs7", "education"))


In [ ]:
# Spouse highest educational level (Q276) -> spouse_education
display(show_raw_counts(wvs7_raw, "Q276"))
display(show_persona_counts("wvs7", "spouse_education"))


In [ ]:
# Marital status (Q273) -> marital_status
display(show_raw_counts(wvs7_raw, "Q273"))
display(show_persona_counts("wvs7", "marital_status"))


In [ ]:
# Household size (Q270) -> household_size
display(show_raw_counts(wvs7_raw, "Q270"))
display(show_persona_counts("wvs7", "household_size"))


In [ ]:
# Employment status (Q279) -> employment_status
display(show_raw_counts(wvs7_raw, "Q279"))
display(show_persona_counts("wvs7", "employment_status"))


In [ ]:
# Spouse employment status (Q280) -> spouse_employment
display(show_raw_counts(wvs7_raw, "Q280"))
display(show_persona_counts("wvs7", "spouse_employment"))


In [ ]:
# Respondent occupational group (Q281) — locked -> occupation
display(show_raw_counts(wvs7_raw, "Q281"))
display(show_persona_counts("wvs7", "occupation"))


In [ ]:
# Employer sector (Q284) -> employer_sector
display(show_raw_counts(wvs7_raw, "Q284"))
display(show_persona_counts("wvs7", "employer_sector"))


In [ ]:
# Labor union membership (Q97) -> union_member
display(show_raw_counts(wvs7_raw, "Q97"))
display(show_persona_counts("wvs7", "union_member"))


In [ ]:
# Religious denomination (Q289) -> religion
display(show_raw_counts(wvs7_raw, "Q289"))
display(show_persona_counts("wvs7", "religion"))


In [ ]:
# Religious service attendance (Q171) -> religious_attendance
display(show_raw_counts(wvs7_raw, "Q171"))
display(show_persona_counts("wvs7", "religious_attendance"))


In [ ]:
# Preferred political party (Q223) -> political_party
display(show_raw_counts(wvs7_raw, "Q223"))
display(show_persona_counts("wvs7", "political_party"))


In [ ]:
# Left-Right political scale 1-10 (Q240) -> political_ideology
display(show_raw_counts(wvs7_raw, "Q240"))
display(show_persona_counts("wvs7", "political_ideology"))


In [ ]:
# Interview language (S_INTLANGUAGE) — excluded from persona_spec.yaml (survey admin metadata)
display(show_raw_counts(wvs7_raw, "S_INTLANGUAGE"))
display(show_persona_counts("wvs7", "interview_language", standardized=False))


#### Dataset-specific attributes

In [ ]:
# Number of children ever had (Q274) — locked -> children_ever_had
display(show_raw_counts(wvs7_raw, "Q274"))
display(show_persona_counts("wvs7", "children_ever_had"))


In [ ]:
# Town population size (G_TOWNSIZE) — locked -> town_size
display(show_raw_counts(wvs7_raw, "G_TOWNSIZE"))
display(show_persona_counts("wvs7", "town_size"))


In [ ]:
# Settlement type (H_SETTLEMENT) — excluded (non-US wording duplicating `urban_rural` / `town_size`)
display(show_raw_counts(wvs7_raw, "H_SETTLEMENT"))
display(show_persona_counts("wvs7", "urban_rural"))
display(show_persona_counts("wvs7", "town_size"))


In [ ]:
# Respondent immigrant status (Q263) — locked -> immigrant_status
display(show_raw_counts(wvs7_raw, "Q263"))
display(show_persona_counts("wvs7", "immigrant_status"))


In [ ]:
# Mother immigrant status (Q264) — excluded
display(show_raw_counts(wvs7_raw, "Q264"))


In [ ]:
# Father immigrant status (Q265) — excluded
display(show_raw_counts(wvs7_raw, "Q265"))


In [ ]:
# Country of birth: respondent (Q266) — excluded
display(show_raw_counts(wvs7_raw, "Q266", max_rows=20))


In [ ]:
# Country of birth: mother (Q267) — excluded
display(show_raw_counts(wvs7_raw, "Q267", max_rows=20))


In [ ]:
# Country of birth: father (Q268) — excluded
display(show_raw_counts(wvs7_raw, "Q268", max_rows=20))


In [ ]:
# Lives with parents (Q271) — locked -> lives_with_parents
display(show_raw_counts(wvs7_raw, "Q271"))
display(show_persona_counts("wvs7", "lives_with_parents"))


In [ ]:
# Language spoken at home (Q272) — locked -> home_language
display(show_raw_counts(wvs7_raw, "Q272", max_rows=20))
display(show_persona_counts("wvs7", "home_language", max_rows=20))


In [ ]:
# Mother highest education (Q277) — excluded
display(show_raw_counts(wvs7_raw, "Q277"))


In [ ]:
# Father highest education (Q278) — excluded
display(show_raw_counts(wvs7_raw, "Q278"))


In [ ]:
# Spouse occupational group (Q282) — excluded
display(show_raw_counts(wvs7_raw, "Q282"))


In [ ]:
# Father occupational group at age 14 (Q283) — excluded
display(show_raw_counts(wvs7_raw, "Q283"))


In [ ]:
# Chief wage earner in household (Q285) — locked -> chief_wage_earner
display(show_raw_counts(wvs7_raw, "Q285"))
display(show_persona_counts("wvs7", "chief_wage_earner"))


In [ ]:
# Family savings during past year (Q286) — locked -> family_savings
display(show_raw_counts(wvs7_raw, "Q286"))
display(show_persona_counts("wvs7", "family_savings"))


In [ ]:
# Subjective social class (Q287) — locked -> social_class
display(show_raw_counts(wvs7_raw, "Q287"))
display(show_persona_counts("wvs7", "social_class"))


In [ ]:
# Scale of household incomes 1-10 (Q288) — excluded
display(show_raw_counts(wvs7_raw, "Q288"))


In [ ]:
# Detailed religious denomination (Q289CS9) — excluded (covered by `religion` Q289)
display(show_raw_counts(wvs7_raw, "Q289CS9", max_rows=20))
display(show_persona_counts("wvs7", "religion"))


In [ ]:
# National election voting frequency (Q222) — locked -> voting_frequency
display(show_raw_counts(wvs7_raw, "Q222"))
display(show_persona_counts("wvs7", "voting_frequency"))


### 2.3 Grummon2023 (complete)

In [3]:
from data import grummon2023 as g23

_g23_df, _, _ = g23.load_analytic_cohort()
grummon2023_raw = _g23_df.reset_index(drop=True).copy()
grummon2023_raw["gender"] = grummon2023_raw["gender"].map(persona_loaders.GRUMMON_GENDER_LABELS)
grummon2023_raw["educ"] = grummon2023_raw["educ"].map(persona_loaders.GRUMMON_EDUC_LABELS)
grummon2023_raw["income_10cat"] = grummon2023_raw["income_10cat"].map(persona_loaders.GRUMMON_INCOME_LABELS)
grummon2023_raw["latino"] = grummon2023_raw["latino"].map(persona_loaders.GRUMMON_HISPANIC_LABELS)
grummon2023_raw["partyid"] = grummon2023_raw["partyid"].map(persona_loaders.GRUMMON2023_PARTY_LABELS)
grummon2023_raw["NFPuse"] = grummon2023_raw["NFPuse"].map(g23.NFPUSE_MAP)
grummon2023_raw["Race_checkbox"] = persona_loaders._checkbox_race(
    grummon2023_raw, persona_loaders.GRUMMON2023_RACE_COLS, "grummon2023"
)
grummon2023_raw["dietqual"] = grummon2023_raw["dietqual"].map(g23.DIETQUAL_MAP)
grummon2023_raw["healthlit"] = grummon2023_raw["healthlit"].map(g23.MEDHELP_MAP)
grummon2023_raw["vendingfreq_bev"] = grummon2023_raw["vendingfreq_bev"].map(g23.VENDING_FREQ_MAP)
grummon2023_raw["vendingfreq_snack"] = grummon2023_raw["vendingfreq_snack"].map(g23.VENDING_FREQ_MAP)
grummon2023 = all_personas[all_personas["dataset"] == "grummon2023"].dropna(axis=1, how="all")
grummon2023.head(2)


,dataset,CASEID,country,gender,age,family_income,political_party,household_size,education,race,children_in_household,hispanic,nutrition_label_use,diet_quality,medical_reading_help,bmi
42320,grummon2023,grummon2023_0001,United States,Woman,32.0,"$35,000 to $49,999",Strong Democrat,2.0,Associate's degree,Black or African American,1.0,No,Never,Excellent,Never,Healthy weight (18.5 to <25.0)
42321,grummon2023,grummon2023_0002,United States,Woman,29.0,"$25,000 to $34,999",Independent,5.0,High school graduate (or GED),White,3.0,No,Sometimes,Fair,Sometimes,Obese (>=30.0)


#### Shared attributes

In [4]:
# Constant ("United States") — US-only survey
display(show_persona_counts("grummon2023", "country"))


,count,%
grummon2023 (persona/std): country,,
United States,7945,100.0
<NA>,0,0.0
All,7945,100.0


In [5]:
# Gender (gender) -> gender
display(show_raw_counts(grummon2023_raw, "gender"))
display(show_persona_counts("grummon2023", "gender"))


,count,%
raw: gender,,
Woman,4078,51.3
Man,3779,47.6
Non-binary,69,0.9
Prefer to self-describe,19,0.2
<NA>,0,0.0
All,7945,100.0


,count,%
grummon2023 (persona/std): gender,,
Female,4078,51.3
Male,3779,47.6
Non-binary,69,0.9
Other,19,0.2
<NA>,0,0.0
All,7945,100.0


In [6]:
# Age in years (age) -> age
display(show_raw_counts(grummon2023_raw, "age", max_rows=20))
display(show_persona_counts("grummon2023", "age", max_rows=20))


,count,%
raw: age,,
22.0,199,2.5
35.0,173,2.2
30.0,160,2.0
24.0,159,2.0
59.0,155,2.0
52.0,153,1.9
33.0,151,1.9
32.0,147,1.9
45.0,147,1.9


,count,%
grummon2023 (persona/std): age,,
65+,1699,21.4
25-34,1365,17.2
55-64,1328,16.7
45-54,1294,16.3
35-44,1279,16.1
18-24,980,12.3
<NA>,0,0.0
All,7945,100.0


In [7]:
# Education (educ) -> education
display(show_raw_counts(grummon2023_raw, "educ"))
display(show_persona_counts("grummon2023", "education"))


,count,%
raw: educ,,
Bachelor's degree,1935,24.4
High school graduate (or GED),1872,23.6
Some college or technical school,1814,22.8
Associate's degree,1191,15.0
Graduate or professional degree,888,11.2
Less than high school,245,3.1
<NA>,0,0.0
All,7945,100.0


,count,%
grummon2023 (persona/std): education,,
Bachelor's degree,1935,24.4
High school graduate,1872,23.6
Some college or technical school,1814,22.8
Associate's degree,1191,15.0
Graduate or professional degree,888,11.2
Less than high school,245,3.1
<NA>,0,0.0
All,7945,100.0


In [8]:
# Household income (income_10cat) -> family_income
display(show_raw_counts(grummon2023_raw, "income_10cat"))
display(show_persona_counts("grummon2023", "family_income"))


,count,%
raw: income_10cat,,
"$50,000 to $74,999",1506,19.0
"$35,000 to $49,999",1215,15.3
"$25,000 to $34,999",1101,13.9
"$75,000 to $99,999",868,10.9
"$15,000 to $24,999",847,10.7
"$100,000 to $149,999",760,9.6
"Less than $10,000",735,9.3
"$10,000 to $14,999",460,5.8
"$150,000 to $199,999",254,3.2


,count,%
grummon2023 (persona/std): family_income,,
"$50,000-$74,999",1506,19.0
"$35,000-$49,999",1215,15.3
"$25,000-$34,999",1101,13.9
"$75,000-$99,999",868,10.9
"$15,000-$24,999",847,10.7
"$100,000-$149,999",760,9.6
"Less than $10,000",735,9.3
"$10,000-$14,999",460,5.8
"$150,000-$199,999",254,3.2


In [9]:
# Hispanic/Latino origin (latino) -> hispanic
display(show_raw_counts(grummon2023_raw, "latino"))
display(show_persona_counts("grummon2023", "hispanic"))


,count,%
raw: latino,,
No,6647,83.7
Yes,1297,16.3
<NA>,1,0.0
All,7945,100.0


,count,%
grummon2023 (persona/std): hispanic,,
No,6647,83.7
Yes,1297,16.3
<NA>,1,0.0
All,7945,100.0


In [10]:
# Race checkboxes (Race_1..Race_6) -> race
display(show_raw_counts(grummon2023_raw, "Race_checkbox"))
display(show_persona_counts("grummon2023", "race"))


,count,%
raw: Race_checkbox,,
White,6062,76.3
Black or African American,962,12.1
Another race,300,3.8
Asian,196,2.5
American Indian or Alaska Native,116,1.5
American Indian or Alaska Native; White,81,1.0
Black or African American; White,53,0.7
Asian; White,37,0.5
Native Hawaiian or Other Pacific Islander,36,0.5


,count,%
grummon2023 (persona/std): race,,
White,6062,76.3
Black or African American,962,12.1
Other,300,3.8
Asian,196,2.5
American Indian or Alaska Native,116,1.5
American Indian or Alaska Native; White,81,1.0
Black or African American; White,53,0.7
Asian; White,37,0.5
Native Hawaiian or Other Pacific Islander,36,0.5


In [11]:
# Household size (hhsize_num) -> household_size
display(show_raw_counts(grummon2023_raw, "hhsize_num"))
display(show_persona_counts("grummon2023", "household_size"))


,count,%
raw: hhsize_num,,
2.0,2772,34.9
1.0,1761,22.2
3.0,1476,18.6
4.0,1092,13.7
5.0,515,6.5
6.0,193,2.4
7.0,76,1.0
8.0,38,0.5
9.0,10,0.1


,count,%
grummon2023 (persona/std): household_size,,
2,2772,34.9
1,1761,22.2
3,1476,18.6
4,1092,13.7
5,515,6.5
6,193,2.4
7 or more,135,1.7
<NA>,1,0.0
All,7945,100.0


In [12]:
# Children in household (children_num) -> children_in_household
display(show_raw_counts(grummon2023_raw, "children_num"))
display(show_persona_counts("grummon2023", "children_in_household"))


,count,%
raw: children_num,,
0.0,3745,47.1
1.0,1193,15.0
2.0,781,9.8
3.0,311,3.9
4.0,90,1.1
5.0,35,0.4
6.0,10,0.1
7.0,5,0.1
9.0,2,0.0


,count,%
grummon2023 (persona/std): children_in_household,,
0,3745,47.1
1,1193,15.0
2,781,9.8
3,311,3.9
4,90,1.1
5,35,0.4
7 or more,14,0.2
6,10,0.1
<NA>,1766,22.2


In [13]:
# Political party ID (partyid) -> political_party
display(show_raw_counts(grummon2023_raw, "partyid"))
display(show_persona_counts("grummon2023", "political_party"))


,count,%
raw: partyid,,
Independent,2129,26.8
Strong Democrat,1396,17.6
Strong Republican,1104,13.9
Moderate Democrat,949,11.9
Moderate Republican,771,9.7
Lean Democrat,730,9.2
Lean Republican,595,7.5
Something else,269,3.4
<NA>,2,0.0


,count,%
grummon2023 (persona/std): political_party,,
Independent,2129,26.8
Strong Democrat,1396,17.6
Strong Republican,1104,13.9
Moderate Democrat,949,11.9
Moderate Republican,771,9.7
Lean Democrat,730,9.2
Lean Republican,595,7.5
Other,269,3.4
<NA>,2,0.0


In [14]:
# Nutrition Facts label use (NFPuse; shared with grummon2025) -> nutrition_label_use
display(show_raw_counts(grummon2023_raw, "NFPuse"))
display(show_persona_counts("grummon2023", "nutrition_label_use"))


,count,%
raw: NFPuse,,
Sometimes,3009,37.9
Most of the time,1833,23.1
Rarely,1416,17.8
Never,904,11.4
Always,782,9.8
<NA>,1,0.0
All,7945,100.0


,count,%
grummon2023 (persona/std): nutrition_label_use,,
Sometimes,3009,37.9
Most of the time,1833,23.1
Rarely,1416,17.8
Never,904,11.4
Always,782,9.8
<NA>,1,0.0
All,7945,100.0


#### Dataset-specific attributes

In [15]:
# Prompt: "In general, how healthy is your overall diet? Would you say…"
display(show_raw_counts(grummon2023_raw, "dietqual"))
display(show_persona_counts("grummon2023", "diet_quality"))  # locked -> diet_quality


,count,%
raw: dietqual,,
Good,3216,40.5
Fair,2356,29.7
Very good,1463,18.4
Excellent,465,5.9
Poor,445,5.6
<NA>,0,0.0
All,7945,100.0


,count,%
grummon2023 (persona/std): diet_quality,,
Good,3216,40.5
Fair,2356,29.7
Very good,1463,18.4
Excellent,465,5.9
Poor,445,5.6
<NA>,0,0.0
All,7945,100.0


In [17]:
# Prompt: "How often do you need to have someone help you when you read instructions, pamphlets, or other written materials from your doctor or pharmacy?"
display(show_raw_counts(grummon2023_raw, "healthlit"))
display(show_persona_counts("grummon2023", "medical_reading_help"))  # locked -> medical_reading_help


,count,%
raw: healthlit,,
Never,4201,52.9
Rarely,1908,24.0
Sometimes,1224,15.4
Most of the time,382,4.8
Always,230,2.9
<NA>,0,0.0
All,7945,100.0


,count,%
grummon2023 (persona/std): medical_reading_help,,
Never,4201,52.9
Rarely,1908,24.0
Sometimes,1224,15.4
Most of the time,382,4.8
Always,230,2.9
<NA>,0,0.0
All,7945,100.0


In [18]:
# Prompt: "In a typical week, how often do you buy beverages from vending machines?"
# Excluded: narrow study-specific shopping frequency (~70% floor, 86% identical to snack frequency).
display(show_raw_counts(grummon2023_raw, "vendingfreq_bev"))


,count,%
raw: vendingfreq_bev,,
Never or less than 1 time per week,5548,69.8
1 to 2 times per week,1613,20.3
3 to 4 times per week,592,7.5
5 or more times per week,192,2.4
<NA>,0,0.0
All,7945,100.0


In [19]:
# Prompt: "In a typical week, how often do you buy snacks from vending machines?"
# Excluded: narrow study-specific shopping frequency (~71% floor, 86% identical to beverage frequency).
display(show_raw_counts(grummon2023_raw, "vendingfreq_snack"))


,count,%
raw: vendingfreq_snack,,
Never or less than 1 time per week,5656,71.2
1 to 2 times per week,1470,18.5
3 to 4 times per week,615,7.7
5 or more times per week,181,2.3
<NA>,23,0.3
All,7945,100.0


In [20]:
# Derived BMI category from height & weight prompts:
#   "How tall are you? Please enter your height in feet and inches."
#   "How much do you weigh? Enter your weight in pounds."
display(grummon2023_raw[["height_feet", "height_inches", "weight_lbs"]].describe().round(1))
display(show_persona_counts("grummon2023", "bmi"))  # locked -> bmi


,height_feet,height_inches,weight_lbs
count,7935.0,7932.0,7945.0
mean,5.1,5.3,179.6
std,0.4,3.4,52.4
min,3.0,0.0,40.0
25%,5.0,2.0,145.0
50%,5.0,5.0,173.0
75%,5.0,8.0,207.0
max,8.0,11.0,900.0


,count,%
grummon2023 (persona/std): bmi,,
Healthy weight (18.5 to <25.0),2720,34.2
Obese (>=30.0),2514,31.6
Overweight (25.0 to <30.0),2385,30.0
Underweight (<18.5),312,3.9
<NA>,14,0.2
All,7945,100.0


### 2.4 Grummon2025 (complete)

In [ ]:
from data import grummon2025 as g25

grummon2025_raw = g25.load_analytic_cohort().reset_index(drop=True).copy()
grummon2025_raw["gender"] = grummon2025_raw["gender"].map(persona_loaders.GRUMMON_GENDER_LABELS)
grummon2025_raw["educ"] = grummon2025_raw["educ"].map(persona_loaders.GRUMMON_EDUC_LABELS)
grummon2025_raw["income"] = grummon2025_raw["income"].map(persona_loaders.GRUMMON_INCOME_LABELS)
grummon2025_raw["latino"] = grummon2025_raw["latino"].map(persona_loaders.GRUMMON_HISPANIC_LABELS)
grummon2025_raw["nfpuse"] = grummon2025_raw["nfpuse"].map(g25.NFPUSE_MAP)
grummon2025_raw["race_checkbox"] = persona_loaders._checkbox_race(
    grummon2025_raw, persona_loaders.GRUMMON2025_RACE_COLS, "grummon2025"
)
grummon2025_raw["snap"] = grummon2025_raw["snap"].map(g25.SNAP_MAP)
grummon2025_raw["wic"] = grummon2025_raw["wic"].map(g25.WIC_MAP)
grummon2025_raw["onlineamt"] = grummon2025_raw["onlineamt"].map(g25.ONLINE_MAP)
grummon2025_raw["scr_litall"] = grummon2025_raw["scr_litall"].map(g25.HEALTHLIT_MAP)
grummon2025 = all_personas[all_personas["dataset"] == "grummon2025"].dropna(axis=1, how="all")
grummon2025.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("grummon2025", "country"))


In [ ]:
# Gender (gender) -> gender
display(show_raw_counts(grummon2025_raw, "gender"))
display(show_persona_counts("grummon2025", "gender"))


In [ ]:
# Age in years (age) -> age
display(show_raw_counts(grummon2025_raw, "age", max_rows=20))
display(show_persona_counts("grummon2025", "age", max_rows=20))


In [ ]:
# Education (educ) -> education
display(show_raw_counts(grummon2025_raw, "educ"))
display(show_persona_counts("grummon2025", "education"))


In [ ]:
# Household income (income) -> family_income
display(show_raw_counts(grummon2025_raw, "income"))
display(show_persona_counts("grummon2025", "family_income"))


In [ ]:
# Hispanic/Latino origin (latino) -> hispanic
display(show_raw_counts(grummon2025_raw, "latino"))
display(show_persona_counts("grummon2025", "hispanic"))


In [ ]:
# Race checkboxes (race_1..race_6) -> race
display(show_raw_counts(grummon2025_raw, "race_checkbox"))
display(show_persona_counts("grummon2025", "race"))


In [ ]:
# Household size (hhsize_num) -> household_size
display(show_raw_counts(grummon2025_raw, "hhsize_num"))
display(show_persona_counts("grummon2025", "household_size"))


In [ ]:
# Children in household (children_num) -> children_in_household
display(show_raw_counts(grummon2025_raw, "children_num"))
display(show_persona_counts("grummon2025", "children_in_household"))


In [ ]:
# Nutrition Facts label use (nfpuse; shared with grummon2023) -> nutrition_label_use
display(show_raw_counts(grummon2025_raw, "nfpuse"))
display(show_persona_counts("grummon2025", "nutrition_label_use"))


#### Dataset-specific attributes

In [11]:
# Prompt: "Do you or anyone else in your household currently get SNAP (Supplemental Nutrition Assistance Program) or Food Stamps? This includes any SNAP benefits or Food Stamps, even if the amount is small and even if the benefits are received on behalf of children in the household."
display(show_raw_counts(grummon2025_raw, "snap"))
display(show_persona_counts("grummon2025", "snap_benefits"))  # locked -> snap_benefits


,count,%
raw: snap,,
No,4773,84.7
Yes,817,14.5
Don't know,29,0.5
<NA>,17,0.3
All,5636,100.0


,count,%
grummon2025 (persona/std): snap_benefits,,
No,4773,84.7
Yes,817,14.5
Don't know,29,0.5
<NA>,17,0.3
All,5636,100.0


In [12]:
# Prompt: "Do you or anyone else in your household currently receive WIC benefits?"
display(show_raw_counts(grummon2025_raw, "wic"))
display(show_persona_counts("grummon2025", "wic_benefits"))  # locked -> wic_benefits


,count,%
raw: wic,,
No,5434,96.4
Yes,138,2.4
Don't know,47,0.8
<NA>,17,0.3
All,5636,100.0


,count,%
grummon2025 (persona/std): wic_benefits,,
No,5434,96.4
Yes,138,2.4
Don't know,47,0.8
<NA>,17,0.3
All,5636,100.0


In [13]:
# Prompt: "The next question is about shopping for groceries online. This includes any groceries you buy through the internet or a mobile app for delivery or pick-up, including from Amazon Fresh, Walmart.com, Instacart, Target.com or your local grocery store. In the past year, about what percentage of your groceries did you buy online?"
display(show_raw_counts(grummon2025_raw, "onlineamt"))
display(show_persona_counts("grummon2025", "online_grocery_share"))  # locked -> online_grocery_share


,count,%
raw: onlineamt,,
Less than 25%,3136,55.6
25-49%,991,17.6
75% or more,808,14.3
50-74%,685,12.2
<NA>,16,0.3
All,5636,100.0


,count,%
grummon2025 (persona/std): online_grocery_share,,
Less than 25%,3136,55.6
25-49%,991,17.6
75% or more,808,14.3
50-74%,685,12.2
<NA>,16,0.3
All,5636,100.0


In [14]:
# Derived sum (0-4 correct) of 4 Nutrition Facts label literacy questions:
#   1. "If you eat the entire container, how many calories will you eat? Type the number of calories in the box."
#   2. "If you are allowed to eat 60 grams of carbohydrates as a snack, how much ice cream (in cups) could you have? Type the number of cups in the box."
#   3. "Your doctor advises you to reduce the amount of saturated fat in your diet. You usually have 42 g of saturated fat each day, which includes one serving of ice cream. If you stop eating ice cream, how many grams of saturated fat would you be consuming each day? Type the number of grams in the box."
#   4. "If you usually eat 2,500 calories in a day, what percentage of your daily value of calories will you be eating if you eat one serving?"
display(show_raw_counts(grummon2025_raw, "scr_litall"))
display(show_persona_counts("grummon2025", "nutrition_literacy_score"))  # locked -> nutrition_literacy_score


,count,%
raw: scr_litall,,
4 out of 4,3257,57.8
3 out of 4,1361,24.1
2 out of 4,544,9.7
1 out of 4,327,5.8
0 out of 4,147,2.6
<NA>,0,0.0
All,5636,100.0


,count,%
grummon2025 (persona/std): nutrition_literacy_score,,
4 out of 4,3257,57.8
3 out of 4,1361,24.1
2 out of 4,544,9.7
1 out of 4,327,5.8
0 out of 4,147,2.6
<NA>,0,0.0
All,5636,100.0


### 2.5 Wu2025 (complete)

In [ ]:
from data.wu2025 import stimulus as wu_s

wu2025_raw = wu_s.extract_respondent_df(wu_s.load_raw_df()).replace("", float("nan")).reset_index(drop=True)
wu2025 = all_personas[all_personas["dataset"] == "wu2025"].dropna(axis=1, how="all")
wu2025.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("wu2025", "country"))


In [ ]:
# US state (state) -> state
display(show_raw_counts(wu2025_raw, "state", max_rows=20))
display(show_persona_counts("wu2025", "state", max_rows=20))


In [ ]:
# Sex (sex) -> gender
display(show_raw_counts(wu2025_raw, "sex"))
display(show_persona_counts("wu2025", "gender"))


In [ ]:
# Age bracket (age) -> age
display(show_raw_counts(wu2025_raw, "age"))
display(show_persona_counts("wu2025", "age"))


In [ ]:
# Race (race) -> race
display(show_raw_counts(wu2025_raw, "race"))
display(show_persona_counts("wu2025", "race"))


In [ ]:
# Hispanic/Latino ethnicity (ethnicity) -> hispanic
display(show_raw_counts(wu2025_raw, "ethnicity"))
display(show_persona_counts("wu2025", "hispanic"))


In [ ]:
# Education (education) -> education
display(show_raw_counts(wu2025_raw, "education"))
display(show_persona_counts("wu2025", "education"))


In [ ]:
# Household income (income) -> family_income
display(show_raw_counts(wu2025_raw, "income"))
display(show_persona_counts("wu2025", "family_income"))


In [ ]:
# Political party (partisan folded with republicanstrength, democratstrength, leaner below) -> political_party
display(show_raw_counts(wu2025_raw, "partisan"))
display(show_persona_counts("wu2025", "political_party"))


In [ ]:
# Political ideology (ideology) -> political_ideology
display(show_raw_counts(wu2025_raw, "ideology"))
display(show_persona_counts("wu2025", "political_ideology"))


In [ ]:
# Marital status (marriage) -> marital_status
display(show_raw_counts(wu2025_raw, "marriage"))
display(show_persona_counts("wu2025", "marital_status"))


In [ ]:
# Employment status (employment) -> employment_status
display(show_raw_counts(wu2025_raw, "employment"))
display(show_persona_counts("wu2025", "employment_status"))


In [ ]:
# Residence type (residence) -> residence_type
display(show_raw_counts(wu2025_raw, "residence"))
display(show_persona_counts("wu2025", "residence_type"))


In [ ]:
# Prompt: "Which of the following best describes your sexual orientation?"
display(show_raw_counts(wu2025_raw, "orientation"))
display(show_persona_counts("wu2025", "sexual_orientation"))  # locked -> sexual_orientation


#### Dataset-specific attributes

In [ ]:
# Prompt: "When it comes to politics, would you describe yourself as libertarian, socialist, communist, liberal, conservative, centrist, or something else?"
display(show_raw_counts(wu2025_raw, "ideology_original"))
display(show_persona_counts("wu2025", "political_label"))  # locked -> political_label


In [ ]:
# Prompt: "Do you own your primary residence?"
display(show_raw_counts(wu2025_raw, "residenceownership"))
display(show_persona_counts("wu2025", "owns_home"))  # locked -> owns_home


In [ ]:
# Prompt: "Do you rent your primary residence?"
display(show_raw_counts(wu2025_raw, "residencerental"))
display(show_persona_counts("wu2025", "rents_home"))  # locked -> rents_home


In [ ]:
# Prompt: "Have you ever served on active duty in the US Armed Forces?"
display(show_raw_counts(wu2025_raw, "military"))
display(show_persona_counts("wu2025", "military_service"))  # locked -> military_service


In [ ]:
# Prompt: "Does your household use electricity for heat?"
display(show_raw_counts(wu2025_raw, "householdheat"))
display(show_persona_counts("wu2025", "electric_heating"))  # locked -> electric_heating


In [ ]:
# Prompt: "Would you call yourself a strong Republican or a not very strong Republican?" (Republicans only)
display(show_raw_counts(wu2025_raw, "republicanstrength"))
display(show_persona_counts("wu2025", "political_party"))  # folded -> political_party


In [ ]:
# Prompt: "Would you call yourself a strong Democrat or a not very strong Democrat?" (Democrats only)
display(show_raw_counts(wu2025_raw, "democratstrength"))
display(show_persona_counts("wu2025", "political_party"))  # folded -> political_party


In [ ]:
# Prompt: "Do you think of yourself as closer to the Republican or Democratic party?" (Independents/Other only)
display(show_raw_counts(wu2025_raw, "leaner"))
display(show_persona_counts("wu2025", "political_party"))  # folded -> political_party


In [ ]:
# Prompt: "Generally, how interested are you in politics?"
display(show_raw_counts(wu2025_raw, "polint"))
display(show_persona_counts("wu2025", "political_interest"))  # locked -> political_interest


In [ ]:
# Prompt: "Is your electricity bill included directly in your rent?" (renters only)
display(show_raw_counts(wu2025_raw, "electricityinrent"))
display(show_persona_counts("wu2025", "electricity_included_in_rent"))  # locked -> electricity_included_in_rent


In [ ]:
# Prompt: "Do you know how much your monthly electricity bill is?"
display(show_raw_counts(wu2025_raw, "billknowledge"))
display(show_persona_counts("wu2025", "knows_electricity_bill"))  # locked -> knows_electricity_bill


In [ ]:
# Prompt: "Does your household receive energy from an investor-owned utility (IOU)?"
display(show_raw_counts(wu2025_raw, "currentiou"))
display(show_persona_counts("wu2025", "uses_investor_owned_utility"))  # locked -> uses_investor_owned_utility


In [ ]:
# Prompt: "Do you currently participate in a Community Choice Aggregation (CCA)?"
display(show_raw_counts(wu2025_raw, "currentcca"))
display(show_persona_counts("wu2025", "uses_community_choice_aggregation"))  # locked -> uses_community_choice_aggregation


In [ ]:
# Prompt: "Do you know what a Community Choice Aggregation (CCA) is?"
display(show_raw_counts(wu2025_raw, "ccaknowledge"))
display(show_persona_counts("wu2025", "knows_community_choice_aggregation"))  # locked -> knows_community_choice_aggregation


### 2.6 HINTS 7 (complete)

In [ ]:
_h7_single_cols = [
    "BirthSex", "Age", "TotalHousehold", "ChildrenInHH", "Hisp_Cat",
    "PoliticalViewpoint", "MaritalStatus",
    "CENSREG", "CENSDIV", "NCHSURCODE2013", "PR_RUCA_2010", "RUC2023",
    "Education", "EducB", "EducA", "IncomeRanges", "HHInc", "IncomeRanges_IMP",
    "IncomeFeelings", "Race_Cat2", "RaceEthn", "RaceEthn5", "Occupation2_Cat",
    "WorkHrsPerWeek", "SexualOrientation2", "SexualOrientation_Recode",
    "EthnicGroupBelonging", "Language_Flag", "FormType",
]
_h7_r10_cols = [c for cols, _ in persona_loaders.HINTS7_R10_RACE_GROUPS for c in cols]
_h7_occ_groups = [
    (("Occupation2_Employed",), "Employed"),
    (("Occupation2_OOW1Y",), "Out of work >=1 yr"),
    (("Occupation2_OOWLess1Y",), "Out of work <1 yr"),
    (("Occupation2_Homemaker",), "Homemaker"),
    (("Occupation2_Student",), "Student"),
    (("Occupation2_Retired",), "Retired"),
    (("Occupation2_Disabled",), "Disabled"),
    (("Occupation2_Other",), "Other"),
]
_h7_occ_cols = [c for cols, _ in _h7_occ_groups for c in cols]
with zipfile.ZipFile(persona_loaders.DATA_DIR / persona_loaders.HINTS7_ZIP_REL) as _zf:
    _h7_bytes = _zf.read(persona_loaders.HINTS7_DTA_MEMBER)
with StataReader(io.BytesIO(_h7_bytes)) as _r:
    _h7_vl = _r.value_labels()
    _h7_c2l = dict(zip(_r._varlist, _r._lbllist, strict=True))
_h7_raw_num = pd.read_stata(
    io.BytesIO(_h7_bytes),
    columns=list(dict.fromkeys([*_h7_single_cols, *_h7_r10_cols, *_h7_occ_cols])),
    convert_categoricals=False,
)
hints7_raw = pd.DataFrame({
    c: persona_loaders._decode_stata_col(_h7_raw_num[c], c, _h7_vl, _h7_c2l)
    for c in _h7_single_cols
})
hints7_raw["HINTS_R10_RACE"] = persona_loaders._hints7_multiselect(
    _h7_raw_num, persona_loaders.HINTS7_R10_RACE_GROUPS, "hints7.raw_race"
)
hints7_raw["HINTS_R5_OCCUPATION"] = persona_loaders._hints7_multiselect(
    _h7_raw_num, _h7_occ_groups, "hints7.raw_occ"
)
hints7 = all_personas[all_personas["dataset"] == "hints7"].dropna(axis=1, how="all")
hints7.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("hints7", "country"))


In [ ]:
# Sex assigned at birth (BirthSex) -> gender
display(show_raw_counts(hints7_raw, "BirthSex"))
display(show_persona_counts("hints7", "gender"))


In [ ]:
# Age in years (Age) -> age
display(show_raw_counts(hints7_raw, "Age", max_rows=20))
display(show_persona_counts("hints7", "age", max_rows=20))


In [ ]:
# Region candidate: 9 US Census divisions (matches ISSP US_REG) — locked -> region
display(show_raw_counts(hints7_raw, "CENSDIV"))
display(show_persona_counts("hints7", "region"))


In [ ]:
# Urban/rural candidate: NCHS 2013 6-level metro/fringe/rural — locked -> urban_rural
display(show_raw_counts(hints7_raw, "NCHSURCODE2013"))
display(show_persona_counts("hints7", "urban_rural"))


In [ ]:
# Education candidate: raw R8 (7 levels incl. vocational/technical) — locked -> education
display(show_raw_counts(hints7_raw, "Education"))
display(show_persona_counts("hints7", "education"))


In [ ]:
# Income candidate: raw R16 (9 brackets, observed only) — locked -> family_income
display(show_raw_counts(hints7_raw, "IncomeRanges"))
display(show_persona_counts("hints7", "family_income"))


In [ ]:
# Race candidate: multi-select from raw R10 checkboxes (separate from hispanic) — locked -> race
display(show_raw_counts(hints7_raw, "HINTS_R10_RACE", max_rows=20))
display(show_persona_counts("hints7", "race", max_rows=20))


In [ ]:
# Hispanic/Latino origin (Hisp_Cat) -> hispanic
display(show_raw_counts(hints7_raw, "Hisp_Cat"))
display(show_persona_counts("hints7", "hispanic"))


In [ ]:
# Marital status (MaritalStatus) -> marital_status
display(show_raw_counts(hints7_raw, "MaritalStatus"))
display(show_persona_counts("hints7", "marital_status"))


In [ ]:
# Total household size (TotalHousehold) -> household_size
display(show_raw_counts(hints7_raw, "TotalHousehold"))
display(show_persona_counts("hints7", "household_size"))


In [ ]:
# Children under 18 in household (ChildrenInHH) -> children_in_household
display(show_raw_counts(hints7_raw, "ChildrenInHH"))
display(show_persona_counts("hints7", "children_in_household"))


In [ ]:
# Employment candidate: single-select recode (687 in Multiple) — locked -> employment_status
display(show_raw_counts(hints7_raw, "Occupation2_Cat"))
display(show_persona_counts("hints7", "employment_status"))


In [ ]:
# Political viewpoint (PoliticalViewpoint) -> political_ideology
display(show_raw_counts(hints7_raw, "PoliticalViewpoint"))
display(show_persona_counts("hints7", "political_ideology"))


In [ ]:
# Derived sexual orientation recode — locked -> sexual_orientation
display(show_raw_counts(hints7_raw, "SexualOrientation_Recode"))
display(show_persona_counts("hints7", "sexual_orientation"))


#### Dataset-specific & excluded candidates

In [ ]:
# Region candidate: 4 US Census regions — dropped (using CENSDIV)
display(show_raw_counts(hints7_raw, "CENSREG"))
display(show_persona_counts("hints7", "region"))


In [ ]:
# Urban/rural candidate: USDA 2010 RUCA 4-level — dropped (using NCHSURCODE2013)
display(show_raw_counts(hints7_raw, "PR_RUCA_2010"))
display(show_persona_counts("hints7", "urban_rural"))


In [ ]:
# Urban/rural candidate: USDA 2023 9-level continuum — dropped (using NCHSURCODE2013)
display(show_raw_counts(hints7_raw, "RUC2023"))
display(show_persona_counts("hints7", "urban_rural"))


In [ ]:
# Education candidate: derived 5 standard US levels — excluded (using raw Education)
display(show_raw_counts(hints7_raw, "EducB"))
display(show_persona_counts("hints7", "education"))


In [ ]:
# Education candidate: derived 4 coarse levels — excluded (using raw Education)
display(show_raw_counts(hints7_raw, "EducA"))
display(show_persona_counts("hints7", "education"))


In [ ]:
# Income candidate: derived 6 brackets — excluded (using raw IncomeRanges)
display(show_raw_counts(hints7_raw, "HHInc"))
display(show_persona_counts("hints7", "family_income"))


In [ ]:
# Income candidate: hot-deck imputed 9 brackets (not observed) — dropped (hot-deck imputed)
display(show_raw_counts(hints7_raw, "IncomeRanges_IMP"))
display(show_persona_counts("hints7", "family_income"))


In [ ]:
# Section R17: subjective feelings about present income — dropped (subjective financial stress)
display(show_raw_counts(hints7_raw, "IncomeFeelings"))


In [ ]:
# Race candidate: NCI 12-level race recode (keeps Asian origins separate) — dropped (using HINTS_R10_RACE)
display(show_raw_counts(hints7_raw, "Race_Cat2"))
display(show_persona_counts("hints7", "race", max_rows=20))


In [ ]:
# Race candidate: 7-level combined race + Hispanic recode — dropped (combined with hispanic)
display(show_raw_counts(hints7_raw, "RaceEthn"))
display(show_persona_counts("hints7", "race", max_rows=20))


In [ ]:
# Race candidate: 5-level combined race + Hispanic recode — dropped (combined with hispanic)
display(show_raw_counts(hints7_raw, "RaceEthn5"))
display(show_persona_counts("hints7", "race", max_rows=20))


In [ ]:
# Employment candidate: multi-select from raw R5 checkboxes — excluded (using Occupation2_Cat)
display(show_raw_counts(hints7_raw, "HINTS_R5_OCCUPATION", max_rows=20))
display(show_persona_counts("hints7", "employment_status"))


In [ ]:
# Section R4: hours worked per week — excluded
display(show_raw_counts(hints7_raw, "WorkHrsPerWeek", max_rows=20))


In [ ]:
# Section R12: raw sexual orientation — excluded (using SexualOrientation_Recode)
display(show_raw_counts(hints7_raw, "SexualOrientation2"))
display(show_persona_counts("hints7", "sexual_orientation"))


In [ ]:
# Section R11: sense of belonging to ethnic/racial group (attitudinal) — excluded
display(show_raw_counts(hints7_raw, "EthnicGroupBelonging"))


In [ ]:
# Admin: survey language (English/Spanish/Mixed) — excluded
display(show_raw_counts(hints7_raw, "Language_Flag"))


In [ ]:
# Admin: survey mode (Paper vs Web) — excluded
display(show_raw_counts(hints7_raw, "FormType"))


### 2.7 Covid (complete)

In [ ]:
from data.covid import stimulus as covid_s

_cov_dta = persona_loaders.DATA_DIR / "covid" / "dataverse_files" / "Kreps_etal_vax_replication_data.dta"
_cov_vl, _cov_c2l = persona_loaders._read_stata_value_labels(_cov_dta)
_cov_resp = covid_s.extract_respondent_df(covid_s.load_raw_conjoint_df(_cov_dta)).reset_index(drop=True)
covid_raw = pd.DataFrame({
    c: persona_loaders._decode_stata_col(_cov_resp[c], c, _cov_vl, _cov_c2l)
    if c in _cov_c2l and _cov_c2l[c] in _cov_vl else _cov_resp[c]
    for c in _cov_resp.columns
})
covid_raw["race_checkbox"] = persona_loaders._checkbox_race(_cov_resp, persona_loaders.COVID_RACE_COLS, "covid")
covid = all_personas[all_personas["dataset"] == "covid"].dropna(axis=1, how="all")
covid.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("covid", "country"))


In [ ]:
# US state (state) -> state
display(show_raw_counts(covid_raw, "state", max_rows=20))
display(show_persona_counts("covid", "state", max_rows=20))


In [ ]:
# Gender (gender) -> gender
display(show_raw_counts(covid_raw, "gender"))
display(show_persona_counts("covid", "gender"))


In [ ]:
# Age in years (age) -> age
display(show_raw_counts(covid_raw, "age", max_rows=20))
display(show_persona_counts("covid", "age", max_rows=20))


In [ ]:
# Education (education) -> education
display(show_raw_counts(covid_raw, "education"))
display(show_persona_counts("covid", "education"))


In [ ]:
# Income (income) -> income
display(show_raw_counts(covid_raw, "income"))
display(show_persona_counts("covid", "income"))


In [ ]:
# Race checkboxes (race_1..race_6) -> race
display(show_raw_counts(covid_raw, "race_checkbox"))
display(show_persona_counts("covid", "race"))


In [ ]:
# Religion (relig) -> religion
display(show_raw_counts(covid_raw, "relig"))
display(show_persona_counts("covid", "religion"))


In [23]:
# Prompt: "Would you describe yourself as a born-again or evangelical Christian, or not?" (asked of Christians)
# Prompt to use: "Born-again or evangelical Christian"
display(show_raw_counts(covid_raw, "evangelical"))
display(show_persona_counts("covid", "evangelical_christian"))  # locked -> evangelical_christian


,count,%
raw: evangelical,,
No,551,28.0
Yes,435,22.1
<NA>,985,50.0
All,1971,100.0


,count,%
covid (persona/std): evangelical_christian,,
No,551,28.0
Yes,435,22.1
<NA>,985,50.0
All,1971,100.0


In [ ]:
# Political party ID (party3) -> political_party
display(show_raw_counts(covid_raw, "party3"))
display(show_persona_counts("covid", "political_party"))


In [ ]:
# Political ideology (ideology) -> political_ideology
display(show_raw_counts(covid_raw, "ideology"))
display(show_persona_counts("covid", "political_ideology"))


In [22]:
# Prompt: "Do you currently have health insurance?"
display(show_raw_counts(covid_raw, "insurance"))
display(show_persona_counts("covid", "health_insurance"))  # locked -> health_insurance


,count,%
raw: insurance,,
Yes,1672,84.8
No,244,12.4
Prefer not to say,55,2.8
<NA>,0,0.0
All,1971,100.0


,count,%
covid (persona/std): health_insurance,,
Yes,1672,84.8
No,244,12.4
Prefer not to say,55,2.8
<NA>,0,0.0
All,1971,100.0


In [27]:
# Prompt: "How often in the past have you gotten a flu vaccination?"
display(show_raw_counts(covid_raw, "flu_vaccine"))
display(show_persona_counts("covid", "flu_vaccination_frequency"))  # locked -> flu_vaccination_frequency


,count,%
raw: flu_vaccine,,
Once or twice,569,28.9
Every year,569,28.9
Never,418,21.2
Most years,415,21.1
<NA>,0,0.0
All,1971,100.0


,count,%
covid (persona/std): flu_vaccination_frequency,,
Once or twice,569,28.9
Every year,569,28.9
Never,418,21.2
Most years,415,21.1
<NA>,0,0.0
All,1971,100.0


#### Dataset-specific attributes

In [24]:
# Prompt: "Have you experienced any recent changes in your employment status?"
# Prompt to use: Recent changes in employment status
display(show_raw_counts(covid_raw, "work_status"))
display(show_persona_counts("covid", "recent_employment_change"))  # locked -> recent_employment_change


,count,%
raw: work_status,,
No change,801,40.6
Reduced hours,462,23.4
Getting laid off,270,13.7
Prefer not to say,54,2.7
<NA>,384,19.5
All,1971,100.0


,count,%
covid (persona/std): recent_employment_change,,
No change,801,40.6
Reduced hours,462,23.4
Getting laid off,270,13.7
Prefer not to say,54,2.7
<NA>,384,19.5
All,1971,100.0


In [25]:
# Prompt: "Are you able to work from home?"
display(show_raw_counts(covid_raw, "work_home"))
display(show_persona_counts("covid", "able_to_work_from_home"))  # locked -> able_to_work_from_home


,count,%
raw: work_home,,
Yes,772,39.2
No,487,24.7
To some extent,205,10.4
Prefer not to say,30,1.5
<NA>,477,24.2
All,1971,100.0


,count,%
covid (persona/std): able_to_work_from_home,,
Yes,772,39.2
No,487,24.7
To some extent,205,10.4
Prefer not to say,30,1.5
<NA>,477,24.2
All,1971,100.0


In [26]:
# Prompt: "As of today, do you lean more toward the Democratic Party or the Republican Party?"
# Excluded: fielded without skip logic (70 cross-party Democrat/Republican contradictions vs party3).
display(show_raw_counts(covid_raw, "party_lean"))
display(show_persona_counts("covid", "political_party"))


,count,%
raw: party_lean,,
Democratic Party,848,43.0
Republican Party,809,41.0
Neither/don't know,314,15.9
<NA>,0,0.0
All,1971,100.0


,count,%
covid (persona/std): political_party,,
Democrat,737,37.4
Republican,675,34.2
Independent,436,22.1
Other/don't know,123,6.2
<NA>,0,0.0
All,1971,100.0


In [28]:
# Prompt: "Do you personally know someone who has been officially tested positive for Covid-19?"
# use original wording
display(show_raw_counts(covid_raw, "personal_contact1"))
display(show_persona_counts("covid", "knows_someone_tested_positive_covid"))  # locked -> knows_someone_tested_positive_covid


,count,%
raw: personal_contact1,,
No,1131,57.4
Yes,840,42.6
<NA>,0,0.0
All,1971,100.0


,count,%
covid (persona/std): knows_someone_tested_positive_covid,,
No,1131,57.4
Yes,840,42.6
<NA>,0,0.0
All,1971,100.0


In [29]:
# Prompt: "Do you personally know someone who has been hospitalized or died as a result of having COVID-19?"
# use original wording
display(show_raw_counts(covid_raw, "personal_contact2"))
display(show_persona_counts("covid", "knows_someone_hospitalized_or_died_covid"))  # locked -> knows_someone_hospitalized_or_died_covid


,count,%
raw: personal_contact2,,
No,1485,75.3
Yes,486,24.7
<NA>,0,0.0
All,1971,100.0


,count,%
covid (persona/std): knows_someone_hospitalized_or_died_covid,,
No,1485,75.3
Yes,486,24.7
<NA>,0,0.0
All,1971,100.0


### 2.7b Kreps2021 (complete)

In [ ]:
with zipfile.ZipFile(persona_loaders.DATA_DIR / "kreps2021" / "dataverse_files.zip") as _zf:
    _k21_bytes = _zf.read("vax_misinfo_replication_data_raw.dta")
with StataReader(io.BytesIO(_k21_bytes)) as _r:
    _k21_vl = _r.value_labels()
    _k21_c2l = dict(zip(_r._varlist, _r._lbllist, strict=True))
_k21_race_cols = [c for c, _ in persona_loaders.COVID_RACE_COLS]
_k21_cols = [
    "state", "gender", "age", "education", "relig", "ideology", "party3",
    "party_lean", "income", "insurance", "evangelical", "flu_vaccine",
    "pres_approval", "tv_news", "paper_news", "social_news",
    "sns1", "sns2", "sns3", "sns4", "sns5", "sns6", "sns7",
]
_k21_num = pd.read_stata(
    io.BytesIO(_k21_bytes),
    columns=list(dict.fromkeys([*_k21_cols, *_k21_race_cols])),
    convert_categoricals=False,
)
kreps2021_raw = pd.DataFrame({
    c: persona_loaders._decode_stata_col(_k21_num[c], c, _k21_vl, _k21_c2l)
    for c in _k21_cols
})
kreps2021_raw["race_checkbox"] = persona_loaders._checkbox_race(
    _k21_num, persona_loaders.COVID_RACE_COLS, "kreps2021"
)
kreps2021 = all_personas[all_personas["dataset"] == "kreps2021"].dropna(axis=1, how="all")
kreps2021.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("kreps2021", "country"))


In [ ]:
# US state (state) -> state
display(show_raw_counts(kreps2021_raw, "state", max_rows=20))
display(show_persona_counts("kreps2021", "state", max_rows=20))


In [ ]:
# Gender (gender) -> gender
display(show_raw_counts(kreps2021_raw, "gender"))
display(show_persona_counts("kreps2021", "gender"))


In [ ]:
# Age in years (age) -> age
display(show_raw_counts(kreps2021_raw, "age", max_rows=20))
display(show_persona_counts("kreps2021", "age", max_rows=20))


In [ ]:
# Education (education) -> education
display(show_raw_counts(kreps2021_raw, "education"))
display(show_persona_counts("kreps2021", "education"))


In [ ]:
# Income (income) -> income
display(show_raw_counts(kreps2021_raw, "income"))
display(show_persona_counts("kreps2021", "income"))


In [ ]:
# Race checkboxes (race_1..race_6) -> race
display(show_raw_counts(kreps2021_raw, "race_checkbox"))
display(show_persona_counts("kreps2021", "race"))


In [ ]:
# Religion (relig) -> religion
display(show_raw_counts(kreps2021_raw, "relig"))
display(show_persona_counts("kreps2021", "religion"))


In [ ]:
# Born-again or evangelical Christian (evangelical; shared with covid) -> evangelical_christian
display(show_raw_counts(kreps2021_raw, "evangelical"))
display(show_persona_counts("kreps2021", "evangelical_christian"))


In [ ]:
# Political party ID (party3) -> political_party
display(show_raw_counts(kreps2021_raw, "party3"))
display(show_persona_counts("kreps2021", "political_party"))


In [ ]:
# Political ideology (ideology) -> political_ideology
display(show_raw_counts(kreps2021_raw, "ideology"))
display(show_persona_counts("kreps2021", "political_ideology"))


In [ ]:
# Health insurance (insurance; shared with covid) -> health_insurance
display(show_raw_counts(kreps2021_raw, "insurance"))
display(show_persona_counts("kreps2021", "health_insurance"))


In [28]:
# Prompt: "How often in the past have you gotten a flu vaccination?" (shared with covid)
display(show_raw_counts(kreps2021_raw, "flu_vaccine"))
display(show_persona_counts("kreps2021", "flu_vaccination_frequency"))  # locked -> flu_vaccination_frequency


,count,%
raw: flu_vaccine,,
Once or twice,348,31.8
Every year,335,30.6
Most years,214,19.5
Never,199,18.2
<NA>,0,0.0
All,1096,100.0


,count,%
kreps2021 (persona/std): flu_vaccination_frequency,,
Once or twice,348,31.8
Every year,335,30.6
Most years,214,19.5
Never,199,18.2
<NA>,0,0.0
All,1096,100.0


In [27]:
# Prompt: "Do you approve or disapprove of the way Donald Trump is handling his job as president?" (shared with kriner2026)
display(show_raw_counts(kreps2021_raw, "pres_approval"))
display(show_persona_counts("kreps2021", "trump_approval"))  # locked -> trump_approval (shared with kriner2026)


,count,%
raw: pres_approval,,
Approve,522,47.6
Disapprove,500,45.6
No opinion,74,6.8
<NA>,0,0.0
All,1096,100.0


,count,%
kreps2021 (persona/std): trump_approval,,
Approve,522,47.6
Disapprove,500,45.6
No opinion,74,6.8
<NA>,0,0.0
All,1096,100.0


#### Dataset-specific attributes

In [29]:
# Prompt: "Watching television news programs that report the day's news"
display(show_raw_counts(kreps2021_raw, "tv_news"))
display(show_persona_counts("kreps2021", "tv_news_consumption"))  # locked -> tv_news_consumption


,count,%
raw: tv_news,,
A great deal,510,46.5
A fair amount,335,30.6
Only a little,168,15.3
Not at all,83,7.6
<NA>,0,0.0
All,1096,100.0


,count,%
kreps2021 (persona/std): tv_news_consumption,,
A great deal,510,46.5
A fair amount,335,30.6
Only a little,168,15.3
Not at all,83,7.6
<NA>,0,0.0
All,1096,100.0


In [30]:
# Prompt: "Reading a newspaper (printed or online version)"
display(show_raw_counts(kreps2021_raw, "paper_news"))
display(show_persona_counts("kreps2021", "newspaper_consumption"))  # locked -> newspaper_consumption


,count,%
raw: paper_news,,
A great deal,329,30.0
A fair amount,313,28.6
Only a little,236,21.5
Not at all,218,19.9
<NA>,0,0.0
All,1096,100.0


,count,%
kreps2021 (persona/std): newspaper_consumption,,
A great deal,329,30.0
A fair amount,313,28.6
Only a little,236,21.5
Not at all,218,19.9
<NA>,0,0.0
All,1096,100.0


In [6]:
# Prompt: "Seeing or reading links to news stories on Facebook or other social media sites"
display(show_raw_counts(kreps2021_raw, "social_news"))
display(show_persona_counts("kreps2021", "social_media_news_consumption"))  # locked -> social_media_news_consumption


,count,%
raw: social_news,,
A great deal,353,32.2
A fair amount,336,30.7
Not at all,209,19.1
Only a little,197,18.0
<NA>,1,0.1
All,1096,100.0


,count,%
kreps2021 (persona/std): social_media_news_consumption,,
A great deal,353,32.2
A fair amount,336,30.7
Not at all,209,19.1
Only a little,197,18.0
<NA>,1,0.1
All,1096,100.0


In [7]:
# Prompt: "I use social media very often every day."
display(show_raw_counts(kreps2021_raw, "sns1"))
display(show_persona_counts("kreps2021", "social_media_daily_use"))  # locked -> social_media_daily_use


,count,%
raw: sns1,,
Strongly agree,488,44.5
Agree,367,33.5
Strongly disagree,96,8.8
Disagree,91,8.3
Unsure,54,4.9
<NA>,0,0.0
All,1096,100.0


,count,%
kreps2021 (persona/std): social_media_daily_use,,
Strongly agree,488,44.5
Agree,367,33.5
Strongly disagree,96,8.8
Disagree,91,8.3
Unsure,54,4.9
<NA>,0,0.0
All,1096,100.0


In [8]:
# Prompt: "I share photos and videos on social media."
display(show_raw_counts(kreps2021_raw, "sns2"))
display(show_persona_counts("kreps2021", "social_media_shares_media"))  # locked -> social_media_shares_media


,count,%
raw: sns2,,
Agree,394,35.9
Strongly agree,315,28.7
Strongly disagree,158,14.4
Disagree,158,14.4
Unsure,71,6.5
<NA>,0,0.0
All,1096,100.0


,count,%
kreps2021 (persona/std): social_media_shares_media,,
Agree,394,35.9
Strongly agree,315,28.7
Strongly disagree,158,14.4
Disagree,158,14.4
Unsure,71,6.5
<NA>,0,0.0
All,1096,100.0


In [9]:
# Prompt: "There are social media accounts that I follow constantly."
display(show_raw_counts(kreps2021_raw, "sns3"))
display(show_persona_counts("kreps2021", "social_media_follows_accounts"))  # locked -> social_media_follows_accounts


,count,%
raw: sns3,,
Agree,410,37.4
Strongly agree,337,30.7
Disagree,141,12.9
Strongly disagree,131,12.0
Unsure,76,6.9
<NA>,1,0.1
All,1096,100.0


,count,%
kreps2021 (persona/std): social_media_follows_accounts,,
Agree,410,37.4
Strongly agree,337,30.7
Disagree,141,12.9
Strongly disagree,131,12.0
Unsure,76,6.9
<NA>,1,0.1
All,1096,100.0


In [10]:
# Prompt: "I have more than one social media account that I use actively."
display(show_raw_counts(kreps2021_raw, "sns4"))
display(show_persona_counts("kreps2021", "social_media_multiple_accounts"))  # locked -> social_media_multiple_accounts


,count,%
raw: sns4,,
Agree,330,30.1
Strongly agree,308,28.1
Strongly disagree,218,19.9
Disagree,170,15.5
Unsure,70,6.4
<NA>,0,0.0
All,1096,100.0


,count,%
kreps2021 (persona/std): social_media_multiple_accounts,,
Agree,330,30.1
Strongly agree,308,28.1
Strongly disagree,218,19.9
Disagree,170,15.5
Unsure,70,6.4
<NA>,0,0.0
All,1096,100.0


In [11]:
# Prompt: "I think that social media is very useful."
display(show_raw_counts(kreps2021_raw, "sns5"))
display(show_persona_counts("kreps2021", "social_media_perceived_usefulness"))  # locked -> social_media_perceived_usefulness


,count,%
raw: sns5,,
Agree,420,38.3
Strongly agree,307,28.0
Unsure,198,18.1
Disagree,97,8.9
Strongly disagree,72,6.6
<NA>,2,0.2
All,1096,100.0


,count,%
kreps2021 (persona/std): social_media_perceived_usefulness,,
Agree,420,38.3
Strongly agree,307,28.0
Unsure,198,18.1
Disagree,97,8.9
Strongly disagree,72,6.6
<NA>,2,0.2
All,1096,100.0


In [12]:
# Prompt: "I like to follow my friends' shares on social media."
display(show_raw_counts(kreps2021_raw, "sns6"))
display(show_persona_counts("kreps2021", "social_media_follows_friends_shares"))  # locked -> social_media_follows_friends_shares


,count,%
raw: sns6,,
Agree,466,42.5
Strongly agree,345,31.5
Disagree,102,9.3
Strongly disagree,93,8.5
Unsure,89,8.1
<NA>,1,0.1
All,1096,100.0


,count,%
kreps2021 (persona/std): social_media_follows_friends_shares,,
Agree,466,42.5
Strongly agree,345,31.5
Disagree,102,9.3
Strongly disagree,93,8.5
Unsure,89,8.1
<NA>,1,0.1
All,1096,100.0


In [13]:
# Prompt: "I usually check out what my friends do on social media."
display(show_raw_counts(kreps2021_raw, "sns7"))
display(show_persona_counts("kreps2021", "social_media_checks_friends_activity"))  # locked -> social_media_checks_friends_activity


,count,%
raw: sns7,,
Agree,444,40.5
Strongly agree,356,32.5
Disagree,106,9.7
Unsure,96,8.8
Strongly disagree,94,8.6
<NA>,0,0.0
All,1096,100.0


,count,%
kreps2021 (persona/std): social_media_checks_friends_activity,,
Agree,444,40.5
Strongly agree,356,32.5
Disagree,106,9.7
Unsure,96,8.8
Strongly disagree,94,8.6
<NA>,0,0.0
All,1096,100.0


### 2.8 Berke2023 (complete)

In [ ]:
from data.berke2023 import stimulus as berke_s

_b_df = berke_s.load_raw_df().reset_index(drop=True)
_b_freq = dict(berke_s.FREQ_OPTIONS)
_b_yn = dict(berke_s.YES_NO_DK_OPTIONS)
berke2023_raw = pd.DataFrame({
    "Q_state": _b_df["Q_state"].map(persona_loaders.BERKE_STATE_LABELS),
    "Q_gender": _b_df["Q_gender"].map(persona_loaders.BERKE_GENDER_LABELS),
    "Q_age": _b_df["Q_age"].map(persona_loaders.BERKE_AGE_LABELS),
    "Q_race": persona_loaders._decode_berke_race(_b_df["Q_race"]),
    "Q_income": _b_df["Q_income"].map(persona_loaders.BERKE_INCOME_LABELS),
    "Q_urban_rural": _b_df["Q_urban_rural"].map(persona_loaders.BERKE_URBAN_LABELS),
    "Q_residence_type": _b_df["Q_residence_type"].map(persona_loaders.BERKE_RESTYPE_LABELS),
    "Q_online_shopping": _b_df["Q_online_shopping"].map(_b_freq),
    "Q_takeout_food": _b_df["Q_takeout_food"].map(_b_freq),
    "Q_online_groceries": _b_df["Q_online_groceries"].map(_b_freq),
    "Q_v_1": _b_df["Q_v_1"].map(_b_yn),
    "Q_v_2": _b_df["Q_v_2"].map(_b_yn),
})
berke2023 = all_personas[all_personas["dataset"] == "berke2023"].dropna(axis=1, how="all")
berke2023.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("berke2023", "country"))


In [ ]:
# US state (Q_state) -> state
display(show_raw_counts(berke2023_raw, "Q_state", max_rows=20))
display(show_persona_counts("berke2023", "state", max_rows=20))


In [ ]:
# Gender (Q_gender) -> gender
display(show_raw_counts(berke2023_raw, "Q_gender"))
display(show_persona_counts("berke2023", "gender"))


In [ ]:
# Age bracket (Q_age) -> age
display(show_raw_counts(berke2023_raw, "Q_age"))
display(show_persona_counts("berke2023", "age"))


In [ ]:
# Race/ethnicity (Q_race) -> race
display(show_raw_counts(berke2023_raw, "Q_race"))
display(show_persona_counts("berke2023", "race"))


In [ ]:
# Household income (Q_income) -> family_income
display(show_raw_counts(berke2023_raw, "Q_income"))
display(show_persona_counts("berke2023", "family_income"))


In [ ]:
# Urban/suburban/rural (Q_urban_rural) -> urban_rural
display(show_raw_counts(berke2023_raw, "Q_urban_rural"))
display(show_persona_counts("berke2023", "urban_rural"))


In [ ]:
# Residence type (Q_residence_type) -> residence_type
display(show_raw_counts(berke2023_raw, "Q_residence_type"))
display(show_persona_counts("berke2023", "residence_type"))


#### Dataset-specific attributes

In [ ]:
# Prompt: "How often (on average) do you make online shopping purchases?"
display(show_raw_counts(berke2023_raw, "Q_online_shopping"))
display(show_persona_counts("berke2023", "online_shopping_frequency"))  # locked -> online_shopping_frequency


In [ ]:
# Prompt: "How often (on average) do you have take-out food delivered to your home?"
display(show_raw_counts(berke2023_raw, "Q_takeout_food"))
display(show_persona_counts("berke2023", "takeout_delivery_frequency"))  # locked -> takeout_delivery_frequency


In [ ]:
# Prompt: "How often (on average) do you purchase your groceries online?"
display(show_raw_counts(berke2023_raw, "Q_online_groceries"))
display(show_persona_counts("berke2023", "online_grocery_frequency"))  # locked -> online_grocery_frequency


In [ ]:
# Prompt: "Suppose there is a company that specializes in delivering prescription medications and offers service in your area, and the total price is similar to local pharmacies. If you needed prescription medications, would you ever order them from this company for delivery to your home?"
# Excluded: conjoint block screener gating CHOICE_v1_* (0/392 `No` saw block).
display(show_raw_counts(berke2023_raw, "Q_v_1"))


In [ ]:
# Prompt: "Would you ever order items from a liquor store for delivery to your home?"
# Excluded: conjoint block screener gating CHOICE_v2_* (0/1,334 `No` saw block).
display(show_raw_counts(berke2023_raw, "Q_v_2"))


### 2.9 Kriner2026 (complete)

In [ ]:
from data.Kriner2026 import stimulus as kriner_s

_k_raw_df, _, _k_vlabels = kriner_s.load_raw_df(persona_loaders.DATA_DIR / "Kriner2026" / "Goldfarb_etal_SAF_JCP_replication.dta")
_k_resp = kriner_s.build_persona_input_df(_k_raw_df, _k_vlabels).reset_index(drop=True)
kriner2026_raw = pd.DataFrame({
    col: _k_resp[col].map(_k_vlabels[col]) if col in _k_vlabels else _k_resp[col]
    for col in _k_resp.columns
})
kriner2026_raw["race_checkbox"] = persona_loaders._checkbox_race(
    _k_resp, persona_loaders.KRINER_RACE_COLS, "kriner2026"
)
kriner2026 = all_personas[all_personas["dataset"] == "kriner2026"].dropna(axis=1, how="all")
kriner2026.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("kriner2026", "country"))


In [ ]:
# US state (state) -> state
display(show_raw_counts(kriner2026_raw, "state", max_rows=20))
display(show_persona_counts("kriner2026", "state", max_rows=20))


In [ ]:
# US Census region (vs_region) -> region
display(show_raw_counts(kriner2026_raw, "vs_region"))
display(show_persona_counts("kriner2026", "region"))


In [ ]:
# Urban/suburban/rural (urban_rural_best) -> urban_rural
display(show_raw_counts(kriner2026_raw, "urban_rural_best"))
display(show_persona_counts("kriner2026", "urban_rural"))


In [ ]:
# Gender (gender) -> gender
display(show_raw_counts(kriner2026_raw, "gender"))
display(show_persona_counts("kriner2026", "gender"))


In [ ]:
# Age in years (age_years) -> age
display(show_raw_counts(kriner2026_raw, "age_years", max_rows=20))
display(show_persona_counts("kriner2026", "age", max_rows=20))


In [ ]:
# Education (education) -> education
display(show_raw_counts(kriner2026_raw, "education"))
display(show_persona_counts("kriner2026", "education"))


In [ ]:
# Income (income) -> income
display(show_raw_counts(kriner2026_raw, "income"))
display(show_persona_counts("kriner2026", "income"))


In [ ]:
# Race checkboxes (native_american..race_6) -> race
display(show_raw_counts(kriner2026_raw, "race_checkbox"))
display(show_persona_counts("kriner2026", "race"))


In [ ]:
# Religion (relig) -> religion
display(show_raw_counts(kriner2026_raw, "relig"))
display(show_persona_counts("kriner2026", "religion"))


In [19]:
# Prompt: "As of today, do you lean more toward the Democratic Party or the Republican Party?" (Independents only)
display(show_raw_counts(kriner2026_raw, "party"))
display(show_raw_counts(kriner2026_raw, "party_lean"))
display(show_persona_counts("kriner2026", "political_party"))  # folded into political_party


,count,%
raw: party,,
Independent,317,31.7
Democrat,315,31.5
Republican,281,28.1
Other/don't know,87,8.7
<NA>,0,0.0
All,1000,100.0


,count,%
raw: party_lean,,
Neither/don't know,150,15.0
Democratic Party,84,8.4
Republican Party,82,8.2
<NA>,684,68.4
All,1000,100.0


,count,%
kriner2026 (persona/std): political_party,,
Democrat,315,31.5
Republican,281,28.1
Independent,151,15.1
Other/don't know,87,8.7
Lean Democrat,84,8.4
Lean Republican,82,8.2
<NA>,0,0.0
All,1000,100.0


In [ ]:
# Political ideology (ideology) -> political_ideology
display(show_raw_counts(kriner2026_raw, "ideology"))
display(show_persona_counts("kriner2026", "political_ideology"))


In [ ]:
# Voted in 2024 general election (vs_vote_2024) -> voted_last_election
display(show_raw_counts(kriner2026_raw, "vs_vote_2024"))
display(show_persona_counts("kriner2026", "voted_last_election"))


In [22]:
# Prompt: "Do you approve or disapprove of the way Donald Trump is handling his job as president?"
display(show_raw_counts(kriner2026_raw, "approve"))
display(show_persona_counts("kriner2026", "trump_approval"))  # locked -> trump_approval (shared with kreps2021)


,count,%
raw: approve,,
Disapprove,521,52.1
Approve,359,35.9
No opinion,120,12.0
<NA>,0,0.0
All,1000,100.0


,count,%
kriner2026 (persona/std): trump_approval,,
Disapprove,521,52.1
Approve,359,35.9
No opinion,120,12.0
<NA>,0,0.0
All,1000,100.0


#### Dataset-specific attributes

In [20]:
# Verasight panel profile (no survey prompt): "2024 presidential general election vote choice"
display(show_raw_counts(kriner2026_raw, "vs_votechoice_2024"))
display(show_persona_counts("kriner2026", "vote_choice_2024"))  # locked -> vote_choice_2024


,count,%
raw: vs_votechoice_2024,,
Kamala Harris,300,30.0
Donald Trump,283,28.3
Someone else,81,8.1
<NA>,336,33.6
All,1000,100.0


,count,%
kriner2026 (persona/std): vote_choice_2024,,
Kamala Harris,300,30.0
Donald Trump,283,28.3
Someone else,81,8.1
<NA>,336,33.6
All,1000,100.0


In [21]:
# Prompt: "Which of the following LEAST describes the place where you live:"
# Excluded: negative-ranking companion to already-locked `urban_rural` (`urban_rural_best`).
display(show_raw_counts(kriner2026_raw, "urban_rural_least"))
display(show_raw_counts(kriner2026_raw, "urban_rural_best"))
display(show_persona_counts("kriner2026", "urban_rural"))  # locked from urban_rural_best


,count,%
raw: urban_rural_least,,
Rural,498,49.8
Urban,351,35.1
Suburban,151,15.1
<NA>,0,0.0
All,1000,100.0


,count,%
raw: urban_rural_best,,
Suburban,476,47.6
Urban,320,32.0
Rural,204,20.4
<NA>,0,0.0
All,1000,100.0


,count,%
kriner2026 (persona/std): urban_rural,,
Suburban,476,47.6
Urban,320,32.0
Rural,204,20.4
<NA>,0,0.0
All,1000,100.0


In [23]:
# Prompt: "From what you know about global climate change or global warming, which one of the following statements comes closest to your opinion?"
# Excluded: domain-specific post-conjoint attitude kept as a target survey question (consistent with wu2025/covid).
display(show_raw_counts(kriner2026_raw, "climate_change_old"))


dataset,kriner2026,All
"Global climate change has been established as a serious problem, and immediate action is necessary.",412,412
There is enough evidence that climate change is taking place and some action should be taken.,307,307
"We don't know enough about global climate change, and more research is necessary before we take any actions.",170,170
Concern about global climate change is exaggerated. No action is necessary.,68,68
Global climate change is not occurring; this is not a real issue.,43,43
<NA>,0,0
All,1000,1000


In [24]:
# Prompt: "New technology has significantly increased the production of sustainable aviation fuels made from renewable biomass and waste resources. These sustainable aviation fuels deliver the performance of petroleum-based jet fuel with a fraction of its carbon footprint. This enables airlines to significantly reduce greenhouse gas emissions. How informed are you about sustainable aviation fuels?"
# Excluded: study-specific item inside SAF conjoint introduction block (catalog_idx = 43); kept as target survey question.
display(show_raw_counts(kriner2026_raw, "saf_familiar_old"))


dataset,kriner2026,All
Not at all informed,544,544
Somewhat informed,237,237
Moderately informed,143,143
Very informed,76,76
<NA>,0,0
All,1000,1000


In [25]:
# Prompt: "How knowledgeable would you say you are about science generally?"
# Excluded: post-quiz subjective 7-point self-rating with bare numeric middle levels; kept as target survey question.
display(show_raw_counts(kriner2026_raw, "sci_know_self"))


dataset,kriner2026,All
Somewhat Knowledgeable,386,386
5,169,169
3,106,106
6,106,106
2,99,99
Very Knowledgeable,78,78
Not at All Knowledgeable,56,56
<NA>,0,0
All,1000,1000


### 2.10 Kantorowicz2026 (complete)

In [ ]:
from data.kantorowicz2026 import stimulus as kant_s

_kant_comp, _kant_laundry = kant_s.load_raw_dfs()
kantorowicz2026_raw = kant_s.extract_respondent_df(_kant_comp, _kant_laundry).reset_index(drop=True)
kantorowicz2026 = all_personas[all_personas["dataset"] == "kantorowicz2026"].dropna(axis=1, how="all")
kantorowicz2026.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("kantorowicz2026", "country"))


In [ ]:
# Gender (gender_dummy) -> gender
display(show_raw_counts(kantorowicz2026_raw, "gender_dummy"))
display(show_persona_counts("kantorowicz2026", "gender"))


In [ ]:
# Age bracket (age_dummy) -> age
display(show_raw_counts(kantorowicz2026_raw, "age_dummy"))
display(show_persona_counts("kantorowicz2026", "age"))


In [ ]:
# Education (education_dummy) -> education
display(show_raw_counts(kantorowicz2026_raw, "education_dummy"))
display(show_persona_counts("kantorowicz2026", "education"))


In [ ]:
# Income (income_dummy) -> income
display(show_raw_counts(kantorowicz2026_raw, "income_dummy"))
display(show_persona_counts("kantorowicz2026", "income"))


#### Excluded candidates

In [ ]:
# No verbatim prompt in paper/SI/archive — undocumented behavioral risk-task classification (Risk averse vs Risk seeking)
# Excluded: undocumented behavioral task unused in paper/replication script.
display(show_raw_counts(kantorowicz2026_raw, "risk_aversion_dummy"))


In [ ]:
# Derived binary CRT score (undocumented cutoff); 3-item CRT prompt:
#   "Please answer the following questions:
#    1. A postcard and a pen cost 150 cents in total. The postcard costs 100 cents more than the pen. How many cents does the pen cost?
#    2. If it takes 3 nurses 3 minutes to measure the blood pressure of 3 patients, how long would it take 300 nurses to measure the blood pressure of 300 patients?
#    3. Sally is making some tea. Every hour, the concentration of the tea doubles. If it takes 8 hours for the tea to be ready, how many hours would it take for the tea to reach half of the final concentration?"
# Excluded: undocumented binarization cutoff and no raw item answers in archive.
display(show_raw_counts(kantorowicz2026_raw, "analytical_thinking_dummy"))


In [ ]:
# No verbatim prompt in paper/SI/archive — undocumented risk-task switching consistency flag (0/1)
# Excluded: undocumented risk-task consistency flag.
display(show_raw_counts(kantorowicz2026_raw, "consistent_switch"))


### 2.11 Magistro2025 (complete)

In [26]:
from data.magistro2025 import stimulus as mag_s

magistro2025_raw = mag_s.load_raw_df(
    persona_loaders.DATA_DIR / "magistro2025" / "dataverse_files" / "clean_AJPS.csv"
).reset_index(drop=True).copy()
magistro2025_raw["region_us"] = magistro2025_raw["region"].where(
    magistro2025_raw["countr"] == "US"
).map(persona_loaders.MAGISTRO_US_REGION_LABELS)
magistro2025_raw["province_ca"] = magistro2025_raw["GeoPC_Region1"].where(
    magistro2025_raw["countr"] == "Canada"
).map(persona_loaders.MAGISTRO_CA_PROVINCE_LABELS)
magistro2025 = all_personas[all_personas["dataset"] == "magistro2025"].dropna(axis=1, how="all")
magistro2025.head(2)


,dataset,CASEID,country,gender,age,region,political_party,education,state,income,magistro2025__party2,magistro2025__education_bin,magistro2025__gender_bin
64720,magistro2025,magistro2025_1,US,Woman,45 to 64,South,Democratic,HS or less,NaN,Low,Democratic,No college,Woman
64721,magistro2025,magistro2025_2,US,Man,18 to 29,South,Other,Some college,NaN,Middle,Democratic,No college,Man


#### Shared attributes

In [28]:
# Country (countr) -> country
display(show_raw_counts(magistro2025_raw, "countr"))
display(show_persona_counts("magistro2025", "country"))


,count,%
raw: countr,,
Canada,3031,50.0
US,3025,50.0
<NA>,0,0.0
All,6056,100.0


,count,%
magistro2025 (persona/std): country,,
Canada,3031,50.0
United States,3025,50.0
<NA>,0,0.0
All,6056,100.0


In [29]:
# US Census region (region where countr=='US') -> region
display(show_raw_counts(magistro2025_raw, "region_us"))
display(show_persona_counts("magistro2025", "region"))


,count,%
raw: region_us,,
South,1212,20.0
West,689,11.4
Midwest,606,10.0
Northeast,518,8.6
<NA>,3031,50.0
All,6056,100.0


,count,%
magistro2025 (persona/std): region,,
South,1212,20.0
West,689,11.4
Midwest,606,10.0
Northeast,518,8.6
<NA>,3031,50.0
All,6056,100.0


In [30]:
# Canadian province (GeoPC_Region1 where countr=='Canada') -> state
display(show_raw_counts(magistro2025_raw, "province_ca"))
display(show_persona_counts("magistro2025", "state"))


,count,%
raw: province_ca,,
Ontario,1144,18.9
Quebec,771,12.7
Alberta,358,5.9
British Columbia,338,5.6
Manitoba,112,1.8
Saskatchewan,88,1.5
Nova Scotia,86,1.4
New Brunswick,67,1.1
Newfoundland and Labrador,49,0.8


,count,%
magistro2025 (persona/std): state,,
Ontario,1144,18.9
Quebec,771,12.7
Alberta,358,5.9
British Columbia,338,5.6
Manitoba,112,1.8
Saskatchewan,88,1.5
Nova Scotia,86,1.4
New Brunswick,67,1.1
Newfoundland and Labrador,49,0.8


In [31]:
# Gender (gender) -> gender
display(show_raw_counts(magistro2025_raw, "gender"))
display(show_persona_counts("magistro2025", "gender"))


,count,%
raw: gender,,
Woman,3277,54.1
Man,2699,44.6
Other,79,1.3
<NA>,1,0.0
All,6056,100.0


,count,%
magistro2025 (persona/std): gender,,
Female,3277,54.1
Male,2699,44.6
Other,79,1.3
<NA>,1,0.0
All,6056,100.0


In [32]:
# Age bracket (age_group) -> age
display(show_raw_counts(magistro2025_raw, "age_group"))
display(show_persona_counts("magistro2025", "age"))


,count,%
raw: age_group,,
45 to 64,2269,37.5
30 to 44,1383,22.8
65 or older,1382,22.8
18 to 29,1022,16.9
<NA>,0,0.0
All,6056,100.0


,count,%
magistro2025 (persona/std): age,,
45-64,2269,37.5
30-44,1383,22.8
65+,1382,22.8
18-29,1022,16.9
<NA>,0,0.0
All,6056,100.0


In [33]:
# Education (education) -> education
display(show_raw_counts(magistro2025_raw, "education"))
display(show_persona_counts("magistro2025", "education"))


,count,%
raw: education,,
HS or less,2374,39.2
Some college,1920,31.7
College grad,1165,19.2
Postgrad,597,9.9
<NA>,0,0.0
All,6056,100.0


,count,%
magistro2025 (persona/std): education,,
High school or less,2374,39.2
Some college,1920,31.7
Bachelor's degree,1165,19.2
Postgraduate,597,9.9
<NA>,0,0.0
All,6056,100.0


In [34]:
# Income (income) -> income
display(show_raw_counts(magistro2025_raw, "income"))
display(show_persona_counts("magistro2025", "income"))


,count,%
raw: income,,
Middle,2996,49.5
High,1173,19.4
Low,1102,18.2
<NA>,785,13.0
All,6056,100.0


,count,%
magistro2025 (persona/std): income,,
Middle,2996,49.5
High,1173,19.4
Low,1102,18.2
<NA>,785,13.0
All,6056,100.0


In [35]:
# Political party: fold party + party2 to recover US Independent leaners (Lean Democrat / Lean Republican) -> political_party
display(show_raw_counts(magistro2025_raw, "party"))
display(show_raw_counts(magistro2025_raw, "party2"))
display(show_persona_counts("magistro2025", "political_party"))


,count,%
raw: party,,
Other,2249,37.1
Democratic,1056,17.4
CPC,786,13.0
Republican,755,12.5
LPC,738,12.2
NDP,472,7.8
<NA>,0,0.0
All,6056,100.0


,count,%
raw: party2,,
Other,1820,30.1
Democratic,1260,20.8
Republican,980,16.2
CPC,786,13.0
LPC,738,12.2
NDP,472,7.8
<NA>,0,0.0
All,6056,100.0


,count,%
magistro2025 (persona/std): political_party,,
Other,1820,30.1
Democrat,1056,17.4
Conservative Party of Canada,786,13.0
Republican,755,12.5
Liberal Party of Canada,738,12.2
New Democratic Party,472,7.8
Lean Republican,225,3.7
Lean Democrat,204,3.4
<NA>,0,0.0


#### Excluded derived recodes

In [27]:
# Derived recode (no separate prompt; codebook: "education binary, factor" College vs No college)
# Excluded: deterministic 2-level collapse of already-locked `education`.
display(show_raw_counts(magistro2025_raw, "education_bin"))
display(show_persona_counts("magistro2025", "education"))


,count,%
raw: education_bin,,
No college,4294,70.9
College,1762,29.1
<NA>,0,0.0
All,6056,100.0


,count,%
magistro2025 (persona/std): education,,
High school or less,2374,39.2
Some college,1920,31.7
Bachelor's degree,1165,19.2
Postgraduate,597,9.9
<NA>,0,0.0
All,6056,100.0


In [79]:
# Derived recode (no separate prompt; codebook: "binary variable for gender, factor" Woman vs Man, dropping Other)
# Excluded: lossy copy of already-locked `gender` (maps Other=79 to <NA>).
display(show_raw_counts(magistro2025_raw, "gender_bin"))
display(show_persona_counts("magistro2025", "gender"))


dataset,magistro2025,All
magistro2025__gender_bin,,
Woman,3277,3277
Man,2699,2699
<NA>,80,80
All,6056,6056


### 2.12 Forsythe2023 (complete)

In [41]:
_f_zip = persona_loaders.DATA_DIR / "Forsythe2023" / "DrivingElectrificationReplication-v1.0.2.zip"
_f_member = "crforsythe-DrivingElectrificationReplication-72b0e47/Estimation/Data/ChoiceData/Pooled/Pooled.xlsx"
with zipfile.ZipFile(_f_zip) as _zf:
    _f_xl = pd.ExcelFile(io.BytesIO(_zf.read(_f_member)))
forsythe2023_raw = pd.concat([_f_xl.parse("demo-car"), _f_xl.parse("demo-suv")], ignore_index=True)
forsythe2023_raw["ZIP_combined"] = forsythe2023_raw["ZIP"].fillna(forsythe2023_raw["Clean ZIP"])
forsythe2023 = all_personas[all_personas["dataset"] == "forsythe2023"].dropna(axis=1, how="all")
forsythe2023.head(2)


/tmp/ipykernel_62041/3232715040.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  forsythe2023_raw["ZIP_combined"] = forsythe2023_raw["ZIP"].fillna(forsythe2023_raw["Clean ZIP"])


,dataset,CASEID,country,forsythe2023__Clean gender,forsythe2023__Clean education,forsythe2023__Clean Income Entry,forsythe2023__Clean livingSit,forsythe2023__Clean householdSize,forsythe2023__Clean Vehicle Type,forsythe2023__Clean elecVehTypeOwned,forsythe2023__Clean covid,forsythe2023__Clean Age,forsythe2023__ZIP_combined
70776,forsythe2023,forsythe2023_1,United States,Woman,4 year university degree (bachelors),"$50,000 to $62,499",Married or living with a partner,3,Car,Conventional,None of the above,44.0,33434.0
70777,forsythe2023,forsythe2023_2,United States,Woman,4 year university degree (bachelors),"$87,500 to $99,999",Married or living with a partner,3,Car,Conventional;Hybrid,None of the above,25.0,63109.0


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("forsythe2023", "country"))


In [42]:
# 5-digit US ZIP (ZIP.fillna(Clean ZIP)) mapped via USPS ZIP prefix ranges (invalid 00000/00746 -> <NA>) -> state
display(show_raw_counts(forsythe2023_raw, "ZIP_combined", max_rows=15))
display(show_persona_counts("forsythe2023", "state", max_rows=20))


,count,%
raw: ZIP_combined,,
17543.0,5,0.3
54601.0,5,0.3
33434.0,4,0.2
15601.0,4,0.2
19422.0,4,0.2
64050.0,4,0.2
91311.0,4,0.2
16066.0,4,0.2
18011.0,3,0.2


,count,%
forsythe2023 (persona/std): state,,
Florida,139,8.5
California,128,7.9
Pennsylvania,114,7.0
Texas,107,6.6
New York,95,5.8
Illinois,69,4.2
North Carolina,66,4.1
Ohio,63,3.9
Georgia,60,3.7


In [43]:
# SI footnote p. 5: "We collect information on the gender identity for respondents" -> gender
display(show_raw_counts(forsythe2023_raw, "Clean gender"))
display(show_persona_counts("forsythe2023", "gender"))


,count,%
raw: Clean gender,,
Woman,835,51.4
Man,779,47.9
I do not wish to answer,7,0.4
Other,4,0.2
Trans,1,0.1
<NA>,0,0.0
All,1626,100.0


,count,%
forsythe2023 (persona/std): gender,,
Female,835,51.4
Male,779,47.9
Prefer not to say,7,0.4
Other,4,0.2
Transgender,1,0.1
<NA>,0,0.0
All,1626,100.0


In [44]:
# SI p. 5: respondent age in exact integer years (18-89) -> age
display(show_raw_counts(forsythe2023_raw, "Clean Age", max_rows=20))
display(show_persona_counts("forsythe2023", "age"))


,count,%
raw: Clean Age,,
70.0,61,3.8
32.0,53,3.3
74.0,52,3.2
66.0,52,3.2
68.0,49,3.0
67.0,46,2.8
69.0,45,2.8
36.0,44,2.7
33.0,42,2.6


,count,%
forsythe2023 (persona/std): age,,
65+,572,35.2
35-44,325,20.0
25-34,299,18.4
45-54,200,12.3
55-64,186,11.4
18-24,41,2.5
<NA>,3,0.2
All,1626,100.0


In [45]:
# SI p. 5: "education level" -> education
display(show_raw_counts(forsythe2023_raw, "Clean education"))
display(show_persona_counts("forsythe2023", "education"))


,count,%
raw: Clean education,,
4 year university degree,555,34.1
Masters degree,310,19.1
Some university education,227,14.0
2 year university or trade school degree,226,13.9
Graduated high school,169,10.4
Doctoral degree,79,4.9
4 year university degree (bachelors),49,3.0
Some secondary education,8,0.5
I do not wish to answer,3,0.2


,count,%
forsythe2023 (persona/std): education,,
Bachelor's degree,604,37.1
Master's degree,310,19.1
Some college,227,14.0
2 year university or trade school degree,226,13.9
High school graduate,169,10.4
Doctoral degree,79,4.9
Less than high school,8,0.5
<NA>,3,0.2
All,1626,100.0


In [46]:
# SI p. 5: "household income" -> family_income
display(show_raw_counts(forsythe2023_raw, "Clean Income Entry"))
display(show_persona_counts("forsythe2023", "family_income"))


,count,%
raw: Clean Income Entry,,
"$50,000 to $62,499",204,12.5
"$62,500 to $74,999",175,10.8
"$37,500 to $49,999",173,10.6
"$87,500 to $99,999",154,9.5
"$75,000 to $87,499",148,9.1
"$100,000 to $119,999",116,7.1
"$30,000 to $37,499",107,6.6
"$140,000 to $159,999",92,5.7
"$120,000 to $139,999",85,5.2


,count,%
forsythe2023 (persona/std): family_income,,
"$50,000-$62,499",204,12.5
"$62,500-$74,999",175,10.8
"$37,500-$49,999",173,10.6
"$87,500-$99,999",154,9.5
"$75,000-$87,499",148,9.1
"$100,000-$119,999",116,7.1
"$30,000-$37,499",107,6.6
"$140,000-$159,999",92,5.7
"$120,000-$139,999",85,5.2


In [47]:
# SI p. 5: "marital status" (Clean livingSit) -> marital_status
display(show_raw_counts(forsythe2023_raw, "Clean livingSit"))
display(show_persona_counts("forsythe2023", "marital_status"))


,count,%
raw: Clean livingSit,,
Married or living with a partner,1048,64.5
Single,318,19.6
Divorced,159,9.8
Widowed,81,5.0
Separated,14,0.9
I do not wish to answer,6,0.4
<NA>,0,0.0
All,1626,100.0


,count,%
forsythe2023 (persona/std): marital_status,,
Married or living with a partner,1048,64.5
Single,318,19.6
Divorced,159,9.8
Widowed,81,5.0
Separated,14,0.9
<NA>,6,0.4
All,1626,100.0


In [48]:
# SI Table S3: "Household Size" -> household_size
display(show_raw_counts(forsythe2023_raw, "Clean householdSize"))
display(show_persona_counts("forsythe2023", "household_size"))


,count,%
raw: Clean householdSize,,
2,668,41.1
1,327,20.1
3,268,16.5
4,229,14.1
5,89,5.5
6,27,1.7
7,11,0.7
7 or more,3,0.2
I do not wish to answer,2,0.1


,count,%
forsythe2023 (persona/std): household_size,,
2,668,41.1
1,327,20.1
3,268,16.5
4,229,14.1
5,89,5.5
6,27,1.7
7 or more,14,0.9
<NA>,4,0.2
All,1626,100.0


#### Dataset-specific attributes

In [52]:
# Column `Clean elecVehTypeOwned`: vehicle powertrain types previously or currently owned (multi-select) -> vehicle_powertrain_owned
display(show_raw_counts(forsythe2023_raw, "Clean elecVehTypeOwned"))
display(show_persona_counts("forsythe2023", "vehicle_powertrain_owned"))


,count,%
raw: Clean elecVehTypeOwned,,
Conventional,1354,83.3
Conventional;Hybrid,162,10.0
Hybrid,34,2.1
I have never owned a vehicle,17,1.0
Conventional;Electric,10,0.6
Conventional;Hybrid;Plug-in Hybrid,9,0.6
Electric,9,0.6
Conventional;Plug-in Hybrid,8,0.5
Conventional;Hybrid;Electric,5,0.3


,count,%
forsythe2023 (persona/std): vehicle_powertrain_owned,,
Conventional,1354,83.3
Conventional; Hybrid,162,10.0
Hybrid,34,2.1
I have never owned a vehicle,17,1.0
Conventional; Electric,10,0.6
Conventional; Hybrid; Plug-in Hybrid,9,0.6
Electric,9,0.6
Conventional; Plug-in Hybrid,8,0.5
Conventional; Electric; Hybrid,5,0.3


In [53]:
# SI p. 12: "We ask individuals in the survey in which of the following ways they were impacted by the COVID-19 pandemic: Permanent job loss, Temporary job loss, Salary reduction, None of the above." -> covid_employment_impact
display(show_raw_counts(forsythe2023_raw, "Clean covid"))
display(show_persona_counts("forsythe2023", "covid_employment_impact"))


,count,%
raw: Clean covid,,
None of the above,1186,72.9
Salary reduction,187,11.5
Temporary job loss,117,7.2
Permanent job loss,69,4.2
Temporary job loss;Salary reduction,42,2.6
Permanent job loss;Salary reduction,12,0.7
Permanent job loss;Temporary job loss;Salary reduction,9,0.6
Permanent job loss;Temporary job loss,3,0.2
<NA>,1,0.1


,count,%
forsythe2023 (persona/std): covid_employment_impact,,
None of the above,1186,72.9
Salary reduction,187,11.5
Temporary job loss,117,7.2
Permanent job loss,69,4.2
Salary reduction; Temporary job loss,42,2.6
Permanent job loss; Salary reduction,12,0.7
Permanent job loss; Salary reduction; Temporary job loss,9,0.6
Permanent job loss; Temporary job loss,3,0.2
<NA>,1,0.1


In [54]:
# Screener / choice segment (SI p. 4: recent/intended buyers of passenger cars vs SUVs)
# Excluded: conjoint block screener routing respondents to demo-car vs demo-suv.
display(show_raw_counts(forsythe2023_raw, "Clean Vehicle Type"))


,count,%
raw: Clean Vehicle Type,,
SUV,877,53.9
Car,749,46.1
<NA>,0,0.0
All,1626,100.0


### 2.13 Twin-2K-500 (complete)

In [ ]:
_t2k_paths = sorted(
    (persona_loaders.DATA_DIR / "Twin-2K-500" / "hf" / "wave_split" / "chunks").glob("wave_persona_chunk_*.parquet")
)
_t2k_rows = []
for _p in _t2k_paths:
    _chunk = pd.read_parquet(_p, columns=["pid", "wave1_3_persona_json"])
    for _, _rec in _chunk.iterrows():
        _demo = json.loads(_rec["wave1_3_persona_json"])[0]
        _row = {"pid": str(_rec["pid"])}
        for _q in _demo["Questions"]:
            _row[_q["QuestionID"]] = _q["Answers"]["SelectedText"]
        _t2k_rows.append(_row)
twin2k_raw = pd.DataFrame(_t2k_rows)
twin2k = all_personas[all_personas["dataset"] == "twin2k"].dropna(axis=1, how="all")
twin2k.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("twin2k", "country"))


In [ ]:
# US Census region (QID11) -> region
display(show_raw_counts(twin2k_raw, "QID11"))
display(show_persona_counts("twin2k", "region"))


In [ ]:
# Gender (QID12) -> gender
display(show_raw_counts(twin2k_raw, "QID12"))
display(show_persona_counts("twin2k", "gender"))


In [ ]:
# Age bracket (QID13) -> age
display(show_raw_counts(twin2k_raw, "QID13"))
display(show_persona_counts("twin2k", "age"))


In [ ]:
# Education (QID14) -> education
display(show_raw_counts(twin2k_raw, "QID14"))
display(show_persona_counts("twin2k", "education"))


In [ ]:
# Race/ethnicity (QID15) -> race
display(show_raw_counts(twin2k_raw, "QID15"))
display(show_persona_counts("twin2k", "race"))


In [ ]:
# US citizenship (QID16) -> citizen
display(show_raw_counts(twin2k_raw, "QID16"))
display(show_persona_counts("twin2k", "citizen"))


In [ ]:
# Marital status (QID17) -> marital_status
display(show_raw_counts(twin2k_raw, "QID17"))
display(show_persona_counts("twin2k", "marital_status"))


In [ ]:
# Religion (QID18) -> religion
display(show_raw_counts(twin2k_raw, "QID18"))
display(show_persona_counts("twin2k", "religion"))


In [ ]:
# Religious attendance (QID19) -> religious_attendance
display(show_raw_counts(twin2k_raw, "QID19"))
display(show_persona_counts("twin2k", "religious_attendance"))


In [ ]:
# Political party ID (QID20) -> political_party
display(show_raw_counts(twin2k_raw, "QID20"))
display(show_persona_counts("twin2k", "political_party"))


In [ ]:
# Household income (QID21) -> family_income
display(show_raw_counts(twin2k_raw, "QID21"))
display(show_persona_counts("twin2k", "family_income"))


In [ ]:
# Political ideology (QID22) -> political_ideology
display(show_raw_counts(twin2k_raw, "QID22"))
display(show_persona_counts("twin2k", "political_ideology"))


In [ ]:
# Household size (QID23) -> household_size
display(show_raw_counts(twin2k_raw, "QID23"))
display(show_persona_counts("twin2k", "household_size"))


In [ ]:
# Employment status (QID24) -> employment_status
display(show_raw_counts(twin2k_raw, "QID24"))
display(show_persona_counts("twin2k", "employment_status"))


### 2.14 SubPOP (complete)

In [ ]:
_sp_rows = {}
for _sub, _fn in [("subpop_eval", "raw_subpop_eval.jsonl"), ("subpop_train", "raw_subpop_train.jsonl")]:
    with open(persona_loaders.DATA_DIR / _sub / _fn, encoding="utf-8") as _f:
        for _line in _f:
            _rec = json.loads(_line)
            _cid = f"SUBPOP/{_rec['attribute']}/{_rec['group']}"
            if _cid not in _sp_rows:
                _sp_rows[_cid] = {"CASEID": _cid, _rec["attribute"]: _rec["group"]}
subpop_raw = pd.DataFrame(_sp_rows.values())
subpop = all_personas[all_personas["dataset"] == "subpop"].dropna(axis=1, how="all")
subpop.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only benchmark
display(show_persona_counts("subpop", "country"))


In [ ]:
# Sex (SEX) -> gender
display(show_raw_counts(subpop_raw, "SEX"))
display(show_persona_counts("subpop", "gender"))


In [ ]:
# US Census region (CREGION) -> region
display(show_raw_counts(subpop_raw, "CREGION"))
display(show_persona_counts("subpop", "region"))


In [ ]:
# Education (EDUCATION) -> education
display(show_raw_counts(subpop_raw, "EDUCATION"))
display(show_persona_counts("subpop", "education"))


In [ ]:
# Income (INCOME) -> income
display(show_raw_counts(subpop_raw, "INCOME"))
display(show_persona_counts("subpop", "income"))


In [ ]:
# Race/ethnicity (RACE) -> race
display(show_raw_counts(subpop_raw, "RACE"))
display(show_persona_counts("subpop", "race"))


In [ ]:
# Religion (RELIG) -> religion
display(show_raw_counts(subpop_raw, "RELIG"))
display(show_persona_counts("subpop", "religion"))


In [ ]:
# Political party (POLPARTY) -> political_party
display(show_raw_counts(subpop_raw, "POLPARTY"))
display(show_persona_counts("subpop", "political_party"))


In [ ]:
# Political ideology (POLIDEOLOGY) -> political_ideology
display(show_raw_counts(subpop_raw, "POLIDEOLOGY"))
display(show_persona_counts("subpop", "political_ideology"))


### 2.15 SHED 2025 (complete)

In [ ]:
_sh_cols = [
    *persona_loaders.SHED2025_NUMERIC_RENAME.keys(),
    *persona_loaders.SHED2025_CATEGORICAL_RENAME.keys(),
    "ED0_iflag", "I40_iflag",
]
with zipfile.ZipFile(persona_loaders.DATA_DIR / persona_loaders.SHED2025_ZIP_REL) as _zf:
    _sh_bytes = _zf.read(persona_loaders.SHED2025_DTA_MEMBER)
with StataReader(io.BytesIO(_sh_bytes)) as _r:
    _sh_vl = _r.value_labels()
    _sh_c2l = dict(zip(_r._varlist, _r._lbllist, strict=True))
_sh_num = pd.read_stata(io.BytesIO(_sh_bytes), columns=_sh_cols, convert_categoricals=False)
shed2025_raw = pd.DataFrame({
    c: persona_loaders._decode_stata_col(_sh_num[c], c, _sh_vl, _sh_c2l)
    for c in _sh_cols
})
shed2025 = all_personas[all_personas["dataset"] == "shed2025"].dropna(axis=1, how="all")
shed2025.head(2)


#### Shared attributes

In [ ]:
# Constant ("United States") — US-only survey
display(show_persona_counts("shed2025", "country"))


In [ ]:
# Gender (ppgender) -> gender
display(show_raw_counts(shed2025_raw, "ppgender"))
display(show_persona_counts("shed2025", "gender"))


In [ ]:
# Age in years (ppage) -> age
display(show_raw_counts(shed2025_raw, "ppage", max_rows=20))
display(show_persona_counts("shed2025", "age", max_rows=20))


In [ ]:
# US Census division (ppreg9) -> region
display(show_raw_counts(shed2025_raw, "ppreg9"))
display(show_persona_counts("shed2025", "region"))


In [ ]:
# US state (ppstaten) -> state
display(show_raw_counts(shed2025_raw, "ppstaten", max_rows=20))
display(show_persona_counts("shed2025", "state", max_rows=20))


In [ ]:
# Metro/non-metro status (ppmsacat) -> urban_rural
display(show_raw_counts(shed2025_raw, "ppmsacat"))
display(show_persona_counts("shed2025", "urban_rural"))


In [ ]:
# Education (ED0; loader masks hot-deck ED0_iflag==1 to NaN) -> education
display(show_raw_counts(shed2025_raw, "ED0"))
display(show_persona_counts("shed2025", "education"))


In [ ]:
# Household income (I40; loader masks hot-deck I40_iflag==1 to NaN) -> family_income
display(show_raw_counts(shed2025_raw, "I40"))
display(show_persona_counts("shed2025", "family_income"))


In [ ]:
# Race (ppracem) -> race
display(show_raw_counts(shed2025_raw, "ppracem"))
display(show_persona_counts("shed2025", "race"))


In [ ]:
# Hispanic/Latino origin (pphispan) -> hispanic
display(show_raw_counts(shed2025_raw, "pphispan"))
display(show_persona_counts("shed2025", "hispanic"))


In [ ]:
# US citizenship (ppp20197) -> citizen
display(show_raw_counts(shed2025_raw, "ppp20197"))
display(show_persona_counts("shed2025", "citizen"))


In [ ]:
# Marital status (ppmarit5) -> marital_status
display(show_raw_counts(shed2025_raw, "ppmarit5"))
display(show_persona_counts("shed2025", "marital_status"))


In [ ]:
# Household size (pphhsize) -> household_size
display(show_raw_counts(shed2025_raw, "pphhsize"))
display(show_persona_counts("shed2025", "household_size"))


In [ ]:
# Children under 18 in household (ppkid017) -> children_in_household
display(show_raw_counts(shed2025_raw, "ppkid017"))
display(show_persona_counts("shed2025", "children_in_household"))


In [ ]:
# Employment status (ppemploy) -> employment_status
display(show_raw_counts(shed2025_raw, "ppemploy"))
display(show_persona_counts("shed2025", "employment_status"))


In [ ]:
# Employer sector (ppcm1301) -> employer_sector
display(show_raw_counts(shed2025_raw, "ppcm1301"))
display(show_persona_counts("shed2025", "employer_sector"))


In [ ]:
# Union membership (UNION100) -> union_member
display(show_raw_counts(shed2025_raw, "UNION100"))
display(show_persona_counts("shed2025", "union_member"))


In [ ]:
# Residence type (pphouse4) -> residence_type
display(show_raw_counts(shed2025_raw, "pphouse4"))
display(show_persona_counts("shed2025", "residence_type"))


In [ ]:
# Sexual orientation (pppa_lgb) -> sexual_orientation
display(show_raw_counts(shed2025_raw, "pppa_lgb"))
display(show_persona_counts("shed2025", "sexual_orientation"))
